# TriCheck-LK - Missing Information Detection

This notebook evaluates whether semantic alignment can identify missing
information across English, Sinhala and Tamil government circulars.

Since a labelled dataset of missing multilingual content is not available,
controlled synthetic experiments are performed by removing selected chunks
from otherwise complete official trilingual circulars.

Main steps:
- Establish a complete-document baseline
- Artificially remove a target-language chunk
- Recompute cross-lingual alignment
- Measure changes in semantic similarity
- Identify potential missing-information signals

In [3]:
import pandas as pd
import numpy as np

from sentence_transformers import ( SentenceTransformer, util)


c:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#load chunk dataset

In [12]:
data_file = ( "../data/processed/multilingual_chunks.jsonl")

chunk_df = pd.read_json( data_file, lines=True)
   
print("Dataset shape:" ,chunk_df.shape)
    

Dataset shape: (34316, 8)


#load same embedding model

In [13]:
model_name = (
    "intfloat/"
    "multilingual-e5-small"
)

model = SentenceTransformer(
    model_name
)

print(
    "Model loaded:",
    model_name
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3158.49it/s]


Model loaded: intfloat/multilingual-e5-small


#select same baseline circular

In [14]:
sample_circular_id = 2221

sample_circular = (
    chunk_df[
        chunk_df["circular_id"]
        == sample_circular_id
    ]
    .copy()
)

print(
    sample_circular[
        "language"
    ].value_counts()
)

language
Tamil      28
English    25
Sinhala    25
Name: count, dtype: int64


#language-wise chunks

In [15]:
english_chunks = (
    sample_circular[
        sample_circular["language"] == "English"
    ]
    .sort_values("chunk_index")
    .reset_index(drop=True)
)

sinhala_chunks = (
    sample_circular[
        sample_circular["language"] == "Sinhala"
    ]
    .sort_values("chunk_index")
    .reset_index(drop=True)
)

tamil_chunks = (
    sample_circular[
        sample_circular["language"] == "Tamil"
    ]
    .sort_values("chunk_index")
    .reset_index(drop=True)
)

print("English:", len(english_chunks))
print("Sinhala:", len(sinhala_chunks))
print("Tamil:", len(tamil_chunks))

English: 25
Sinhala: 25
Tamil: 28


#create synthetic missing Sinhala version

In [16]:
missing_chunk_index = 12

sinhala_missing = (
    sinhala_chunks[
        sinhala_chunks["chunk_index"]
        != missing_chunk_index
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Original Sinhala chunks:",
    len(sinhala_chunks)
)

print(
    "After removal:",
    len(sinhala_missing)
)

print(
    "Removed chunk:",
    missing_chunk_index
)

Original Sinhala chunks: 25
After removal: 24
Removed chunk: 12


#Identify removed chunk information

In [17]:
removed_chunk = (
    sinhala_chunks[
        sinhala_chunks["chunk_index"]
        == missing_chunk_index
    ]
    .iloc[0]["chunk_text"]
)

print(removed_chunk)

(ආ) වූහගත් සම්මු පරීක් ලෂණය (i) රාජ්‍ය ලේසේවා ලේකොමිෂන් සභාව විසින් පත් කරනු ලබන සම්මු පරීක් ලෂණ මණ්ඉලයක් ල විසින් අයදුම්කරුවන්ලේේ කළමනාකරණ කුසලත්ාව, නායකත්ව ගුණාංග, සන්නිලේේදන කුසලත්ාව සහ ලේප රුෂත්වය මැන බැලීම සඳහා පහත් සඳහන් පටිපාටිය අනුව වූහගත් සම්මු පරීක් ලෂණයක් ල පවත්වනු ලැලේේ. එහිදී ලබාගත් හැකි පපරිම ලකුණු සං යාව ලකුණු විසිපහ (25) කි. විෂය ක් ලලේෂේත්‍රය පපරිම ලකුණු 01 කළමනාකරණ කුසලත්ාව 10 02 නායකත්ව ගුණාංග 05 03 සන්නිලේේදන කුසලත්ා 05 0/ ලේප රුෂත්වය 05 එකතුව 25 I. එක් ල එක් ල විෂයය සඳහා අවම වශලේයන් සියයට පනහ (50%) ක් ල ලේහෝ ඊට ඉහළ ලකුණු ලබා ගත් අයදුම්කරුවන් අතුරින් සියලුම විෂයයන්ට ලැබූ ලකුණුවල මුළු එකතුව සහ ලේජ්‍යෂ්්ත්වය සඳහා ලැබූ ලකුණු යන ලේදලේකහිම මුළු ලකුණු අනුව ඉහළම ලකුණු ලබා ගන්නා අයදුම්කරුවන් සාමානය සම්මු පරීක් ලෂණයට ලේපනී සිටීම සඳහා සුදුසුකම් ලබයි. සුදුසුකම් ලැබූ අයදුම්කරුවන් අත්රින් සාමානය සම්මු පරීක් ලෂණයට කැඳවනු ලබන්ලේන්, පුරප්පාඩු සං යාව අනුව බඳවා ගැනීමට අලේප්ක් ලත ත් සං යාව හා පුරප්පාඩු ප්‍රමාණලේයන් 25% ක ප්‍රමාණයක් ල යන ලේදලේක් ල එකතුව වශලේයන් ගැලේනන සං යාවක් ල පමණි.


#embeddings for complete and missing versions

In [18]:
# Generate English embeddings

english_embeddings = model.encode(
    [
        "query: " + text
        for text in english_chunks["chunk_text"]
    ],
    normalize_embeddings=True,
    convert_to_tensor=True
)


# Generate embeddings for COMPLETE Sinhala document

sinhala_complete_embeddings = model.encode(
    [
        "query: " + text
        for text in sinhala_chunks["chunk_text"]
    ],
    normalize_embeddings=True,
    convert_to_tensor=True
)


# Generate embeddings for MISSING Sinhala document

sinhala_missing_embeddings = model.encode(
    [
        "query: " + text
        for text in sinhala_missing["chunk_text"]
    ],
    normalize_embeddings=True,
    convert_to_tensor=True
)


print(
    "English embeddings:",
    english_embeddings.shape
)

print(
    "Complete Sinhala embeddings:",
    sinhala_complete_embeddings.shape
)

print(
    "Missing Sinhala embeddings:",
    sinhala_missing_embeddings.shape
)

English embeddings: torch.Size([25, 384])
Complete Sinhala embeddings: torch.Size([25, 384])
Missing Sinhala embeddings: torch.Size([24, 384])


#similarity matrices

In [19]:
complete_similarity = util.cos_sim(
    english_embeddings,
    sinhala_complete_embeddings
)

missing_similarity = util.cos_sim(
    english_embeddings,
    sinhala_missing_embeddings
)


print(
    "Complete similarity matrix:",
    complete_similarity.shape
)

print(
    "Missing similarity matrix:",
    missing_similarity.shape
)

Complete similarity matrix: torch.Size([25, 25])
Missing similarity matrix: torch.Size([25, 24])


# actual missing-information signal measure

In [20]:
def dtw_align(similarity_matrix):

    similarity = (
        similarity_matrix
        .detach()
        .cpu()
        .numpy()
    )

    n, m = similarity.shape

    cost = np.full(
        (n + 1, m + 1),
        np.inf
    )

    cost[0, 0] = 0

    backtrack = np.zeros(
        (n + 1, m + 1),
        dtype=int
    )

    for i in range(1, n + 1):

        for j in range(1, m + 1):

            local_cost = (
                1
                - similarity[
                    i - 1,
                    j - 1
                ]
            )

            previous_costs = [
                cost[i - 1, j - 1],
                cost[i - 1, j],
                cost[i, j - 1]
            ]

            best_move = np.argmin(
                previous_costs
            )

            cost[i, j] = (
                local_cost
                + previous_costs[
                    best_move
                ]
            )

            backtrack[i, j] = best_move

    i = n
    j = m

    path = []

    while i > 0 and j > 0:

        path.append(
            (
                i - 1,
                j - 1
            )
        )

        move = backtrack[i, j]

        if move == 0:
            i -= 1
            j -= 1

        elif move == 1:
            i -= 1

        else:
            j -= 1

    path.reverse()

    return path

In [21]:
#align complete and missing versions
complete_path = dtw_align(
    complete_similarity
)

missing_path = dtw_align(
    missing_similarity
)

print(
    "Complete alignment path:",
    len(complete_path)
)

print(
    "Missing alignment path:",
    len(missing_path)
)

Complete alignment path: 26
Missing alignment path: 26


In [22]:
#best aligned similarity  for each English chunk in DTW
def get_source_coverage(
    path,
    similarity_matrix,
    source_count
):

    coverage = {
        i: []
        for i in range(source_count)
    }

    for source_pos, target_pos in path:

        score = (
            similarity_matrix[
                source_pos,
                target_pos
            ]
            .item()
        )

        coverage[
            source_pos
        ].append(score)

    best_scores = []

    for source_pos in range(
        source_count
    ):

        scores = coverage[
            source_pos
        ]

        if scores:
            best_scores.append(
                max(scores)
            )
        else:
            best_scores.append(
                np.nan
            )

    return best_scores

#compare complete vs missing

In [23]:
complete_scores = get_source_coverage(
    complete_path,
    complete_similarity,
    len(english_chunks)
)

missing_scores = get_source_coverage(
    missing_path,
    missing_similarity,
    len(english_chunks)
)


comparison_df = pd.DataFrame(
    {
        "english_chunk": (
            english_chunks[
                "chunk_index"
            ].tolist()
        ),

        "complete_score": (
            complete_scores
        ),

        "missing_score": (
            missing_scores
        )
    }
)


comparison_df[
    "score_drop"
] = (
    comparison_df[
        "complete_score"
    ]
    -
    comparison_df[
        "missing_score"
    ]
)


comparison_df.sort_values(
    "score_drop",
    ascending=False
).head(10)

,english_chunk,complete_score,missing_score,score_drop
13,13,0.884603,0.859566,0.025037
1,1,0.883450,0.883450,0.000000
2,2,0.840178,0.840178,0.000000
3,3,0.898025,0.898025,0.000000
4,4,0.885152,0.885152,0.000000
5,5,0.878842,0.878842,0.000000
6,6,0.906521,0.906521,0.000000
7,7,0.862912,0.862912,0.000000
0,0,0.900059,0.900059,0.000000
8,8,0.840467,0.840467,0.000000


#verify affected English chunk

In [24]:


# Find which English chunk was aligned
# with the Sinhala chunk that we removed

affected_english_chunks = []

for en_pos, si_pos in complete_path:

    original_si_chunk = (
        sinhala_chunks
        .iloc[si_pos]["chunk_index"]
    )

    if original_si_chunk == missing_chunk_index:

        affected_english_chunks.append(
            english_chunks
            .iloc[en_pos]["chunk_index"]
        )


print(
    "Removed Sinhala chunk:",
    missing_chunk_index
)

print(
    "Corresponding English chunk(s):",
    affected_english_chunks
)

Removed Sinhala chunk: 12
Corresponding English chunk(s): [np.int64(13)]


In [25]:
largest_drop_row = (
    comparison_df
    .sort_values(
        "score_drop",
        ascending=False
    )
    .iloc[0]
)

print(
    "Largest drop detected at English chunk:",
    int(
        largest_drop_row[
            "english_chunk"
        ]
    )
)

print(
    "Score drop:",
    round(
        largest_drop_row[
            "score_drop"
        ],
        4
    )
)

Largest drop detected at English chunk: 13
Score drop: 0.025


#remove 3 consecutive Sinhala chunks

In [26]:
missing_chunk_indices = [
    11,
    12,
    13
]

sinhala_block_missing = (
    sinhala_chunks[
        ~sinhala_chunks[
            "chunk_index"
        ].isin(
            missing_chunk_indices
        )
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Original Sinhala chunks:",
    len(sinhala_chunks)
)

print(
    "After block removal:",
    len(sinhala_block_missing)
)

print(
    "Removed chunks:",
    missing_chunk_indices
)

Original Sinhala chunks: 25
After block removal: 22
Removed chunks: [11, 12, 13]


#embeddings + alignment

In [27]:
block_missing_embeddings = model.encode(
    [
        "query: " + text
        for text in sinhala_block_missing[
            "chunk_text"
        ]
    ],
    normalize_embeddings=True,
    convert_to_tensor=True
)

block_missing_similarity = util.cos_sim(
    english_embeddings,
    block_missing_embeddings
)

block_missing_path = dtw_align(
    block_missing_similarity
)

block_missing_scores = get_source_coverage(
    block_missing_path,
    block_missing_similarity,
    len(english_chunks)
)

#compare with complete version

In [28]:
block_comparison_df = pd.DataFrame(
    {
        "english_chunk":
            english_chunks[
                "chunk_index"
            ].tolist(),

        "complete_score":
            complete_scores,

        "missing_score":
            block_missing_scores
    }
)

block_comparison_df[
    "score_drop"
] = (
    block_comparison_df[
        "complete_score"
    ]
    -
    block_comparison_df[
        "missing_score"
    ]
)

block_comparison_df.sort_values(
    "score_drop",
    ascending=False
).head(10)

,english_chunk,complete_score,missing_score,score_drop
16,16,0.911885,0.854597,0.057288
15,15,0.888772,0.850993,0.037779
14,14,0.858720,0.827482,0.031238
24,24,0.899729,0.872492,0.027237
12,12,0.863737,0.839544,0.024193
13,13,0.884603,0.861875,0.022728
18,18,0.872881,0.864549,0.008333
23,23,0.873470,0.867101,0.006369
19,19,0.840165,0.835263,0.004902
22,22,0.880971,0.880112,0.000859


#removed Sinhala block actually for which English region -correspond verification

In [29]:
affected_english_chunks = []

for en_pos, si_pos in complete_path:

    si_chunk_index = (
        sinhala_chunks
        .iloc[si_pos]["chunk_index"]
    )

    if si_chunk_index in missing_chunk_indices:

        affected_english_chunks.append(
            english_chunks
            .iloc[en_pos]["chunk_index"]
        )


affected_english_chunks = sorted(
    set(affected_english_chunks)
)

print(
    "Removed Sinhala chunks:",
    missing_chunk_indices
)

print(
    "Corresponding English region:",
    affected_english_chunks
)

Removed Sinhala chunks: [11, 12, 13]
Corresponding English region: [np.int64(12), np.int64(13), np.int64(14)]


#complete vs missing DTW mapping around that region

In [30]:
print("COMPLETE ALIGNMENT")

for en_pos, si_pos in complete_path:

    en_index = (
        english_chunks
        .iloc[en_pos]["chunk_index"]
    )

    if 9 <= en_index <= 18:

        si_index = (
            sinhala_chunks
            .iloc[si_pos]["chunk_index"]
        )

        score = (
            complete_similarity[
                en_pos,
                si_pos
            ]
            .item()
        )

        print(
            f"EN {en_index} → SI {si_index}"
            f" | {score:.4f}"
        )


print("\nMISSING ALIGNMENT")

for en_pos, si_pos in block_missing_path:

    en_index = (
        english_chunks
        .iloc[en_pos]["chunk_index"]
    )

    if 9 <= en_index <= 18:

        si_index = (
            sinhala_block_missing
            .iloc[si_pos]["chunk_index"]
        )

        score = (
            block_missing_similarity[
                en_pos,
                si_pos
            ]
            .item()
        )

        print(
            f"EN {en_index} → SI {si_index}"
            f" | {score:.4f}"
        )

COMPLETE ALIGNMENT
EN 9 → SI 8 | 0.8721
EN 10 → SI 9 | 0.8649
EN 11 → SI 10 | 0.8716
EN 12 → SI 11 | 0.8637
EN 13 → SI 12 | 0.8846
EN 14 → SI 13 | 0.8587
EN 15 → SI 14 | 0.8888
EN 16 → SI 15 | 0.9119
EN 17 → SI 16 | 0.8668
EN 18 → SI 17 | 0.8729

MISSING ALIGNMENT
EN 9 → SI 8 | 0.8721
EN 10 → SI 9 | 0.8649
EN 11 → SI 10 | 0.8716
EN 12 → SI 10 | 0.8395
EN 13 → SI 10 | 0.8619
EN 14 → SI 14 | 0.8275
EN 15 → SI 15 | 0.8510
EN 16 → SI 16 | 0.8546
EN 17 → SI 17 | 0.8786
EN 18 → SI 18 | 0.8645


In [31]:
#compare alignment compression
def create_path_dataframe(
    path,
    similarity_matrix,
    source_chunks,
    target_chunks
):

    rows = []

    for source_pos, target_pos in path:

        score = (
            similarity_matrix[
                source_pos,
                target_pos
            ]
            .item()
        )

        rows.append(
            {
                "source_position": source_pos,
                "target_position": target_pos,

                "english_chunk": (
                    source_chunks
                    .iloc[source_pos]["chunk_index"]
                ),

                "target_chunk": (
                    target_chunks
                    .iloc[target_pos]["chunk_index"]
                ),

                "similarity": score
            }
        )

    return pd.DataFrame(rows)

In [32]:
complete_path_df = create_path_dataframe(
    complete_path,
    complete_similarity,
    english_chunks,
    sinhala_chunks
)

missing_path_df = create_path_dataframe(
    block_missing_path,
    block_missing_similarity,
    english_chunks,
    sinhala_block_missing
)

In [33]:
#detect repeated target mappings
def find_compression_runs(path_df):

    df = path_df.copy()

    # New run whenever target position changes
    df["run_id"] = (
        df["target_position"]
        .ne(
            df["target_position"].shift()
        )
        .cumsum()
    )

    runs = (
        df.groupby("run_id")
        .agg(
            target_position=(
                "target_position",
                "first"
            ),

            source_start=(
                "english_chunk",
                "first"
            ),

            source_end=(
                "english_chunk",
                "last"
            ),

            run_length=(
                "english_chunk",
                "size"
            ),

            mean_similarity=(
                "similarity",
                "mean"
            ),

            min_similarity=(
                "similarity",
                "min"
            )
        )
        .reset_index(drop=True)
    )

    return (
        runs[
            runs["run_length"] > 1
        ]
        .sort_values(
            "run_length",
            ascending=False
        )
    )

In [34]:
print("COMPLETE DOCUMENT")
display(
    find_compression_runs(
        complete_path_df
    )
)

print("\nMISSING DOCUMENT")
display(
    find_compression_runs(
        missing_path_df
    )
)

COMPLETE DOCUMENT


,target_position,source_start,source_end,run_length,mean_similarity,min_similarity
6,6,6,7,2,0.884716,0.862912



MISSING DOCUMENT


,target_position,source_start,source_end,run_length,mean_similarity,min_similarity
10,10,11,13,3,0.857678,0.839544
6,6,6,7,2,0.884716,0.862912


In [35]:
#automated synthetic experiment function
def run_missing_experiment(
    removed_indices
):

    # ---------------------------------
    # Remove selected Sinhala chunks
    # ---------------------------------

    target_missing = (
        sinhala_chunks[
            ~sinhala_chunks[
                "chunk_index"
            ].isin(
                removed_indices
            )
        ]
        .copy()
        .reset_index(drop=True)
    )


    # ---------------------------------
    # Generate embeddings
    # ---------------------------------

    target_embeddings = model.encode(
        [
            "query: " + text
            for text in target_missing[
                "chunk_text"
            ]
        ],
        normalize_embeddings=True,
        convert_to_tensor=True
    )


    # ---------------------------------
    # Similarity + DTW alignment
    # ---------------------------------

    similarity_matrix = util.cos_sim(
        english_embeddings,
        target_embeddings
    )

    path = dtw_align(
        similarity_matrix
    )


    # ---------------------------------
    # Source coverage scores
    # ---------------------------------

    missing_scores = get_source_coverage(
        path,
        similarity_matrix,
        len(english_chunks)
    )

    score_drops = (
        np.array(complete_scores)
        -
        np.array(missing_scores)
    )


    # ---------------------------------
    # Alignment compression
    # ---------------------------------

    path_df = create_path_dataframe(
        path,
        similarity_matrix,
        english_chunks,
        target_missing
    )

    compression = find_compression_runs(
        path_df
    )


    if len(compression) > 0:

        max_run_length = int(
            compression[
                "run_length"
            ].max()
        )

        lowest_run_similarity = float(
            compression[
                "min_similarity"
            ].min()
        )

    else:

        max_run_length = 1
        lowest_run_similarity = np.nan


    # ---------------------------------
    # Return experiment summary
    # ---------------------------------

    return {
        "removed_chunks": str(
            removed_indices
        ),

        "removed_count": len(
            removed_indices
        ),

        "max_score_drop": round(
            float(
                np.nanmax(score_drops)
            ),
            4
        ),

        "max_run_length": (
            max_run_length
        ),

        "lowest_run_similarity": round(
            lowest_run_similarity,
            4
        )
        if not np.isnan(
            lowest_run_similarity
        )
        else np.nan
    }

#test different missing sizes

In [36]:
experiments = [
    [5],
    [12],
    [18],

    [7, 8],
    [14, 15],

    [5, 6, 7],
    [11, 12, 13],
    [17, 18, 19]
]


experiment_results = pd.DataFrame(
    [
        run_missing_experiment(
            removed
        )
        for removed in experiments
    ]
)

experiment_results

,removed_chunks,removed_count,max_score_drop,max_run_length,lowest_run_similarity
0,[5],1,0.0091,3,0.8629
1,[12],1,0.0250,2,0.8596
2,[18],1,0.0272,2,0.8629
3,"[7, 8]",2,0.0151,3,0.8253
4,"[14, 15]",2,0.0519,2,0.8616
5,"[5, 6, 7]",3,0.0431,4,0.8365
6,"[11, 12, 13]",3,0.0573,3,0.8395
7,"[17, 18, 19]",3,0.0272,3,0.8501


#Tamil baseline embeddings

In [37]:
tamil_embeddings = model.encode(
    [
        "query: " + text
        for text in tamil_chunks["chunk_text"]
    ],
    normalize_embeddings=True,
    convert_to_tensor=True
)

en_ta_similarity = util.cos_sim(
    english_embeddings,
    tamil_embeddings
)

en_ta_path = dtw_align(
    en_ta_similarity
)

print(
    "English-Tamil matrix:",
    en_ta_similarity.shape
)

print(
    "English-Tamil alignment path:",
    len(en_ta_path)
)

English-Tamil matrix: torch.Size([25, 28])
English-Tamil alignment path: 28


In [38]:
en_ta_scores = get_source_coverage(
    en_ta_path,
    en_ta_similarity,
    len(english_chunks)
)

triangulation_df = pd.DataFrame(
    {
        "english_chunk":
            english_chunks[
                "chunk_index"
            ].tolist(),

        "sinhala_complete_score":
            complete_scores,

        "sinhala_missing_score":
            block_missing_scores,

        "tamil_score":
            en_ta_scores
    }
)

triangulation_df[
    "si_ta_gap"
] = (
    triangulation_df[
        "tamil_score"
    ]
    -
    triangulation_df[
        "sinhala_missing_score"
    ]
)

triangulation_df.sort_values(
    "si_ta_gap",
    ascending=False
).head(10)

,english_chunk,sinhala_complete_score,sinhala_missing_score,tamil_score,si_ta_gap
16,16,0.911885,0.854597,0.897654,0.043057
15,15,0.888772,0.850993,0.887991,0.036998
2,2,0.840178,0.840178,0.870182,0.030004
24,24,0.899729,0.872492,0.889910,0.017419
14,14,0.858720,0.827482,0.844240,0.016759
10,10,0.864876,0.864876,0.875407,0.010531
12,12,0.863737,0.839544,0.846629,0.007085
19,19,0.840165,0.835263,0.840232,0.004969
9,9,0.872149,0.872149,0.872492,0.000343
8,8,0.840467,0.840467,0.839351,-0.001116


#Create evaluation dataset

In [39]:
def add_compression_feature(
    path_df
):

    target_counts = (
        path_df[
            "target_position"
        ]
        .value_counts()
    )

    result = path_df.copy()

    result[
        "compression_count"
    ] = (
        result[
            "target_position"
        ]
        .map(target_counts)
    )

    return result

In [40]:
missing_feature_df = (
    add_compression_feature(
        missing_path_df
    )
)

missing_feature_df[
    [
        "english_chunk",
        "target_chunk",
        "similarity",
        "compression_count"
    ]
].head(20)

,english_chunk,target_chunk,similarity,compression_count
0,0,0,0.900059,1
1,1,1,0.883450,1
2,2,2,0.840178,1
3,3,3,0.898025,1
4,4,4,0.885152,1
5,5,5,0.878842,1
6,6,6,0.906521,2
7,7,6,0.862912,2
8,8,7,0.840467,1
9,9,8,0.872149,1


#anomaly feature table

In [41]:
# One row per English chunk
sinhala_features = (
    missing_feature_df
    .groupby("english_chunk")
    .agg(
        sinhala_similarity=(
            "similarity",
            "max"
        ),
        compression_count=(
            "compression_count",
            "max"
        )
    )
    .reset_index()
)


# Tamil acts as the third-language reference
tamil_features = pd.DataFrame(
    {
        "english_chunk":
            english_chunks[
                "chunk_index"
            ].tolist(),

        "tamil_similarity":
            en_ta_scores
    }
)


feature_df = (
    sinhala_features
    .merge(
        tamil_features,
        on="english_chunk",
        how="left"
    )
)

In [42]:
#cross-language disagreement
feature_df[
    "cross_language_gap"
] = (
    feature_df[
        "tamil_similarity"
    ]
    -
    feature_df[
        "sinhala_similarity"
    ]
)

In [43]:
#local semantic anomaly
feature_df[
    "local_median"
] = (
    feature_df[
        "sinhala_similarity"
    ]
    .rolling(
        window=5,
        center=True,
        min_periods=1
    )
    .median()
)


feature_df[
    "local_drop"
] = (
    feature_df[
        "local_median"
    ]
    -
    feature_df[
        "sinhala_similarity"
    ]
)

In [44]:
#synthetic ground-truth label

feature_df[
    "synthetic_missing"
] = (
    feature_df[
        "english_chunk"
    ]
    .isin(
        affected_english_chunks
    )
)

In [45]:
display(
    feature_df[
        [
            "english_chunk",
            "sinhala_similarity",
            "tamil_similarity",
            "cross_language_gap",
            "compression_count",
            "local_drop",
            "synthetic_missing"
        ]
    ]
    .iloc[9:18]
)

,english_chunk,sinhala_similarity,tamil_similarity,cross_language_gap,compression_count,local_drop,synthetic_missing
9,9,0.872149,0.872492,0.000343,1,-0.007273,False
10,10,0.864876,0.875407,0.010531,1,0.000000,False
11,11,0.871616,0.850310,-0.021306,3,-0.006740,False
12,12,0.839544,0.846629,0.007085,3,0.022330,True
13,13,0.861875,0.836926,-0.024949,3,-0.010882,True
14,14,0.827482,0.844240,0.016759,1,0.023511,True
15,15,0.850993,0.887991,0.036998,1,0.003604,False
16,16,0.854597,0.897654,0.043057,1,0.000000,False
17,17,0.878631,0.834390,-0.044241,1,-0.024034,False


In [46]:
#choose 20 moderate-size circulars
# Count chunks for each circular and language

chunk_counts = (
    chunk_df
    .groupby(
        [
            "circular_id",
            "language"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

chunk_counts.head()

language,English,Sinhala,Tamil
circular_id,,,
3,5,3,5
6,2,2,2
7,2,2,3
8,6,4,6
9,3,2,3


In [47]:
# Select moderate-size circulars
# to keep evaluation computationally manageable

eligible_circulars = chunk_counts[
    chunk_counts["English"].between(8, 25)
    & chunk_counts["Sinhala"].between(8, 25)
    & chunk_counts["Tamil"].between(8, 30)
]

print(
    "Eligible circulars:",
    len(eligible_circulars)
)

Eligible circulars: 144


In [48]:
#reproducible sample
evaluation_ids = (
    eligible_circulars
    .sample(
        n=min(
            20,
            len(eligible_circulars)
        ),
        random_state=42
    )
    .index
    .tolist()
)

print(
    "Evaluation circulars:",
    len(evaluation_ids)
)

print(
    evaluation_ids
)

Evaluation circulars: 20
[2076, 657, 1700, 1821, 1506, 388, 2142, 1567, 1583, 642, 1467, 1649, 1810, 2146, 1899, 1564, 1062, 1598, 2107, 1618]


In [49]:
#generic evaluation function
def build_synthetic_case(
    circular_id
):

    # --------------------------------------------------
    # Get one trilingual circular
    # --------------------------------------------------

    circular = (
        chunk_df[
            chunk_df["circular_id"]
            == circular_id
        ]
        .copy()
    )

    english = (
        circular[
            circular["language"] == "English"
        ]
        .sort_values("chunk_index")
        .reset_index(drop=True)
    )

    sinhala = (
        circular[
            circular["language"] == "Sinhala"
        ]
        .sort_values("chunk_index")
        .reset_index(drop=True)
    )

    tamil = (
        circular[
            circular["language"] == "Tamil"
        ]
        .sort_values("chunk_index")
        .reset_index(drop=True)
    )


    # --------------------------------------------------
    # Generate complete-document embeddings
    # --------------------------------------------------

    en_embeddings = model.encode(
        [
            "query: " + text
            for text in english["chunk_text"]
        ],
        normalize_embeddings=True,
        convert_to_tensor=True
    )

    si_embeddings = model.encode(
        [
            "query: " + text
            for text in sinhala["chunk_text"]
        ],
        normalize_embeddings=True,
        convert_to_tensor=True
    )

    ta_embeddings = model.encode(
        [
            "query: " + text
            for text in tamil["chunk_text"]
        ],
        normalize_embeddings=True,
        convert_to_tensor=True
    )


    # --------------------------------------------------
    # Complete EN-SI alignment
    # --------------------------------------------------

    complete_similarity = util.cos_sim(
        en_embeddings,
        si_embeddings
    )

    complete_path = dtw_align(
        complete_similarity
    )


    # --------------------------------------------------
    # EN-TA reference alignment
    # --------------------------------------------------

    en_ta_similarity = util.cos_sim(
        en_embeddings,
        ta_embeddings
    )

    en_ta_path = dtw_align(
        en_ta_similarity
    )

    tamil_scores = get_source_coverage(
        en_ta_path,
        en_ta_similarity,
        len(english)
    )


    # --------------------------------------------------
    # Remove 3 Sinhala chunks near document middle
    # --------------------------------------------------

    middle_position = (
        len(sinhala) // 2
    )

    remove_positions = [
        middle_position - 1,
        middle_position,
        middle_position + 1
    ]

    removed_indices = (
        sinhala
        .iloc[remove_positions][
            "chunk_index"
        ]
        .tolist()
    )


    sinhala_missing = (
        sinhala[
            ~sinhala[
                "chunk_index"
            ].isin(
                removed_indices
            )
        ]
        .copy()
        .reset_index(drop=True)
    )


    # --------------------------------------------------
    # Find English chunks corresponding to
    # the removed Sinhala content
    # --------------------------------------------------

    affected_english = set()

    for en_pos, si_pos in complete_path:

        si_index = (
            sinhala
            .iloc[si_pos][
                "chunk_index"
            ]
        )

        if si_index in removed_indices:

            affected_english.add(
                int(
                    english
                    .iloc[en_pos][
                        "chunk_index"
                    ]
                )
            )


    # --------------------------------------------------
    # Missing-version embeddings and alignment
    # --------------------------------------------------

    missing_embeddings = model.encode(
        [
            "query: " + text
            for text in sinhala_missing[
                "chunk_text"
            ]
        ],
        normalize_embeddings=True,
        convert_to_tensor=True
    )

    missing_similarity = util.cos_sim(
        en_embeddings,
        missing_embeddings
    )

    missing_path = dtw_align(
        missing_similarity
    )


    # --------------------------------------------------
    # Create alignment features
    # --------------------------------------------------

    path_df = create_path_dataframe(
        missing_path,
        missing_similarity,
        english,
        sinhala_missing
    )

    path_df = add_compression_feature(
        path_df
    )


    sinhala_features = (
        path_df
        .groupby(
            "english_chunk"
        )
        .agg(
            sinhala_similarity=(
                "similarity",
                "max"
            ),

            compression_count=(
                "compression_count",
                "max"
            )
        )
        .reset_index()
    )


    tamil_features = pd.DataFrame(
        {
            "english_chunk":
                english[
                    "chunk_index"
                ].tolist(),

            "tamil_similarity":
                tamil_scores
        }
    )


    feature_df = (
        sinhala_features
        .merge(
            tamil_features,
            on="english_chunk",
            how="left"
        )
        .sort_values(
            "english_chunk"
        )
        .reset_index(drop=True)
    )


    # --------------------------------------------------
    # Cross-language disagreement
    # --------------------------------------------------

    feature_df[
        "cross_language_gap"
    ] = (
        feature_df[
            "tamil_similarity"
        ]
        -
        feature_df[
            "sinhala_similarity"
        ]
    )


    # --------------------------------------------------
    # Local semantic anomaly
    # --------------------------------------------------

    feature_df[
        "local_median"
    ] = (
        feature_df[
            "sinhala_similarity"
        ]
        .rolling(
            window=5,
            center=True,
            min_periods=1
        )
        .median()
    )


    feature_df[
        "local_drop"
    ] = (
        feature_df[
            "local_median"
        ]
        -
        feature_df[
            "sinhala_similarity"
        ]
    )


    # --------------------------------------------------
    # Synthetic ground-truth label
    # --------------------------------------------------

    feature_df[
        "synthetic_missing"
    ] = (
        feature_df[
            "english_chunk"
        ]
        .isin(
            affected_english
        )
    )


    feature_df[
        "circular_id"
    ] = circular_id


    feature_df[
        "removed_chunks"
    ] = str(
        removed_indices
    )


    return feature_df

In [50]:
#Test one NEW circular first
#Use first evaluation ID:
test_case = build_synthetic_case(
    2076
)

print(
    "Rows:",
    len(test_case)
)

print(
    "Missing-labelled chunks:",
    test_case[
        "synthetic_missing"
    ].sum()
)

display(
    test_case[
        [
            "circular_id",
            "english_chunk",
            "sinhala_similarity",
            "tamil_similarity",
            "cross_language_gap",
            "compression_count",
            "local_drop",
            "synthetic_missing"
        ]
    ]
)

Rows: 17
Missing-labelled chunks: 3


,circular_id,english_chunk,sinhala_similarity,tamil_similarity,cross_language_gap,compression_count,local_drop,synthetic_missing
0,2076,0,0.885735,0.874626,-0.011109,1,-0.044293,False
1,2076,1,0.838772,0.844154,0.005382,1,0.001335,False
2,2076,2,0.841442,0.844838,0.003396,1,0.000000,False
3,2076,3,0.826291,0.840490,0.014200,1,0.015152,False
4,2076,4,0.900229,0.873396,-0.026833,1,-0.047465,False
5,2076,5,0.852764,0.843387,-0.009377,1,0.006011,False
6,2076,6,0.859190,0.843828,-0.015362,1,-0.000414,False
7,2076,7,0.858776,0.846989,-0.011787,4,0.000000,True
8,2076,8,0.854844,0.848874,-0.005970,4,0.004345,True
9,2076,9,0.869524,0.837515,-0.032009,4,-0.009733,True


In [51]:
#run all 20 evaluation circulars
all_evaluation_cases = []

for index, circular_id in enumerate(
    evaluation_ids,
    start=1
):

    print(
        f"Processing {index}/"
        f"{len(evaluation_ids)}"
        f" - Circular {circular_id}"
    )

    try:

        case_df = build_synthetic_case(
            circular_id
        )

        all_evaluation_cases.append(
            case_df
        )

    except Exception as error:

        print(
            "Failed:",
            circular_id,
            error
        )

Processing 1/20 - Circular 2076
Processing 2/20 - Circular 657
Processing 3/20 - Circular 1700
Processing 4/20 - Circular 1821
Processing 5/20 - Circular 1506
Processing 6/20 - Circular 388
Processing 7/20 - Circular 2142
Processing 8/20 - Circular 1567
Processing 9/20 - Circular 1583
Processing 10/20 - Circular 642
Processing 11/20 - Circular 1467
Processing 12/20 - Circular 1649
Processing 13/20 - Circular 1810
Processing 14/20 - Circular 2146
Processing 15/20 - Circular 1899
Processing 16/20 - Circular 1564
Processing 17/20 - Circular 1062
Processing 18/20 - Circular 1598
Processing 19/20 - Circular 2107
Processing 20/20 - Circular 1618


In [52]:
#combine
evaluation_df = pd.concat(
    all_evaluation_cases,
    ignore_index=True
)

print(
    "\nEvaluation rows:",
    len(evaluation_df)
)

print(
    "Circulars processed:",
    evaluation_df[
        "circular_id"
    ].nunique()
)

print(
    "\nClass distribution:"
)

print(
    evaluation_df[
        "synthetic_missing"
    ].value_counts()
)


Evaluation rows: 249
Circulars processed: 20

Class distribution:
synthetic_missing
False    183
True      66
Name: count, dtype: int64


In [53]:
#feature differences
feature_columns = [
    "sinhala_similarity",
    "tamil_similarity",
    "cross_language_gap",
    "compression_count",
    "local_drop"
]

feature_summary = (
    evaluation_df
    .groupby(
        "synthetic_missing"
    )[feature_columns]
    .mean()
)

feature_summary

,sinhala_similarity,tamil_similarity,cross_language_gap,compression_count,local_drop
synthetic_missing,,,,,
False,0.851162,0.854547,0.003385,1.693989,0.000240
True,0.841957,0.854532,0.012575,2.712121,0.001359


In [54]:
#Median 
feature_median = (
    evaluation_df
    .groupby(
        "synthetic_missing"
    )[feature_columns]
    .median()
)

feature_median

,sinhala_similarity,tamil_similarity,cross_language_gap,compression_count,local_drop
synthetic_missing,,,,,
False,0.848108,0.852460,0.002164,1.0,0.0
True,0.837750,0.854638,0.010849,3.0,0.0


# classification imports

In [55]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [56]:
#prepare X, y and groups
feature_columns = [
    "sinhala_similarity",
    "tamil_similarity",
    "cross_language_gap",
    "compression_count",
    "local_drop"
]

X = evaluation_df[
    feature_columns
].copy()

y = (
    evaluation_df[
        "synthetic_missing"
    ]
    .astype(int)
)

groups = evaluation_df[
    "circular_id"
]

In [57]:
#split by circular, not by rows
group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_index, test_index = next(
    group_split.split(
        X,
        y,
        groups=groups
    )
)

X_train = X.iloc[train_index]
X_test = X.iloc[test_index]

y_train = y.iloc[train_index]
y_test = y.iloc[test_index]

train_groups = groups.iloc[train_index]
test_groups = groups.iloc[test_index]


print(
    "Training circulars:",
    train_groups.nunique()
)

print(
    "Testing circulars:",
    test_groups.nunique()
)

print(
    "Train rows:",
    len(X_train)
)

print(
    "Test rows:",
    len(X_test)
)

Training circulars: 15
Testing circulars: 5
Train rows: 184
Test rows: 65


In [58]:
#train interpretable Logistic Regression
detector = Pipeline(
    [
        (
            "scaler",
            StandardScaler()
        ),

        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

detector.fit(
    X_train,
    y_train
)

print(
    "Detector trained."
)

Detector trained.


In [59]:
#evaluate
y_pred = detector.predict(
    X_test
)

print(
    "Accuracy:",
    round(
        accuracy_score(
            y_test,
            y_pred
        ),
        4
    )
)

print(
    "Precision:",
    round(
        precision_score(
            y_test,
            y_pred
        ),
        4
    )
)

print(
    "Recall:",
    round(
        recall_score(
            y_test,
            y_pred
        ),
        4
    )
)

print(
    "F1-score:",
    round(
        f1_score(
            y_test,
            y_pred
        ),
        4
    )
)

print(
    "\nConfusion Matrix:"
)

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)

print(
    "\nClassification Report:"
)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Normal",
            "Missing"
        ]
    )
)

Accuracy: 0.6923
Precision: 0.4348
Recall: 0.5882
F1-score: 0.5

Confusion Matrix:
[[35 13]
 [ 7 10]]

Classification Report:
              precision    recall  f1-score   support

      Normal       0.83      0.73      0.78        48
     Missing       0.43      0.59      0.50        17

    accuracy                           0.69        65
   macro avg       0.63      0.66      0.64        65
weighted avg       0.73      0.69      0.71        65



#Check which feature high influence

In [60]:
classifier = detector.named_steps[
    "classifier"
]

feature_importance = pd.DataFrame(
    {
        "feature": feature_columns,
        "coefficient": classifier.coef_[0]
    }
)

feature_importance[
    "absolute_coefficient"
] = (
    feature_importance[
        "coefficient"
    ].abs()
)

feature_importance = (
    feature_importance
    .sort_values(
        "absolute_coefficient",
        ascending=False
    )
)

feature_importance

,feature,coefficient,absolute_coefficient
3,compression_count,0.884524,0.884524
2,cross_language_gap,0.402604,0.402604
0,sinhala_similarity,-0.351165,0.351165
4,local_drop,-0.153150,0.153150
1,tamil_similarity,0.044672,0.044672


In [61]:
#grouped cross-validation on training data
from sklearn.model_selection import (
    GroupKFold,
    cross_val_predict
)

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

In [62]:
feature_sets = {
    "all_features": [
        "sinhala_similarity",
        "tamil_similarity",
        "cross_language_gap",
        "compression_count",
        "local_drop"
    ],

    "top_3": [
        "compression_count",
        "cross_language_gap",
        "sinhala_similarity"
    ],

    "top_2": [
        "compression_count",
        "sinhala_similarity"
    ],

    "compression_only": [
        "compression_count"
    ]
}

In [63]:
#only training circulars with Group 5-fold CV
cv_results = []

group_cv = GroupKFold(
    n_splits=5
)

for name, columns in feature_sets.items():

    model_cv = Pipeline(
        [
            (
                "scaler",
                StandardScaler()
            ),
            (
                "classifier",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=1000,
                    random_state=42
                )
            )
        ]
    )

    probabilities = cross_val_predict(
        model_cv,
        X_train[columns],
        y_train,
        groups=train_groups,
        cv=group_cv,
        method="predict_proba"
    )[:, 1]

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    cv_results.append(
        {
            "feature_set": name,

            "precision": precision_score(
                y_train,
                predictions
            ),

            "recall": recall_score(
                y_train,
                predictions
            ),

            "f1_score": f1_score(
                y_train,
                predictions
            )
        }
    )


cv_results_df = pd.DataFrame(
    cv_results
)

cv_results_df.sort_values(
    "f1_score",
    ascending=False
)

,feature_set,precision,recall,f1_score
3,compression_only,0.586957,0.551020,0.568421
2,top_2,0.476190,0.612245,0.535714
1,top_3,0.450704,0.653061,0.533333
0,all_features,0.418919,0.632653,0.504065


In [64]:
#get grouped out-of-fold probabilities for compression-only
best_features = [
    "compression_count"
]

best_cv_model = Pipeline(
    [
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

best_probabilities = cross_val_predict(
    best_cv_model,
    X_train[best_features],
    y_train,
    groups=train_groups,
    cv=group_cv,
    method="predict_proba"
)[:, 1]

In [65]:
#search threshold using training CV only
threshold_results = []

for threshold in np.arange(
    0.30,
    0.71,
    0.05
):

    predictions = (
        best_probabilities
        >= threshold
    ).astype(int)

    threshold_results.append(
        {
            "threshold": round(
                threshold,
                2
            ),

            "precision": precision_score(
                y_train,
                predictions,
                zero_division=0
            ),

            "recall": recall_score(
                y_train,
                predictions,
                zero_division=0
            ),

            "f1_score": f1_score(
                y_train,
                predictions,
                zero_division=0
            )
        }
    )

threshold_df = pd.DataFrame(
    threshold_results
)

threshold_df.sort_values(
    "f1_score",
    ascending=False
)

,threshold,precision,recall,f1_score
4,0.50,0.586957,0.551020,0.568421
6,0.60,0.586957,0.551020,0.568421
5,0.55,0.586957,0.551020,0.568421
3,0.45,0.432432,0.653061,0.520325
2,0.40,0.432432,0.653061,0.520325
7,0.65,0.588235,0.408163,0.481928
8,0.70,0.612903,0.387755,0.475000
1,0.35,0.276596,0.795918,0.410526
0,0.30,0.260116,0.918367,0.405405


In [66]:
#train final detector on training circulars
final_features = [
    "compression_count"
]

final_detector = Pipeline(
    [
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

final_detector.fit(
    X_train[final_features],
    y_train
)

print(
    "Final detector trained."
)

Final detector trained.


In [67]:
#predict on held-out test circulars
test_probabilities = (
    final_detector
    .predict_proba(
        X_test[final_features]
    )[:, 1]
)

final_threshold = 0.50

final_predictions = (
    test_probabilities
    >= final_threshold
).astype(int)

In [68]:
# final metrics
print(
    "Final Accuracy:",
    round(
        accuracy_score(
            y_test,
            final_predictions
        ),
        4
    )
)

print(
    "Final Precision:",
    round(
        precision_score(
            y_test,
            final_predictions
        ),
        4
    )
)

print(
    "Final Recall:",
    round(
        recall_score(
            y_test,
            final_predictions
        ),
        4
    )
)

print(
    "Final F1-score:",
    round(
        f1_score(
            y_test,
            final_predictions
        ),
        4
    )
)

print(
    "\nConfusion Matrix:"
)

print(
    confusion_matrix(
        y_test,
        final_predictions
    )
)

print(
    "\nClassification Report:"
)

print(
    classification_report(
        y_test,
        final_predictions,
        target_names=[
            "Normal",
            "Missing"
        ]
    )
)

Final Accuracy: 0.6615
Final Precision: 0.3684
Final Recall: 0.4118
Final F1-score: 0.3889

Confusion Matrix:
[[36 12]
 [10  7]]

Classification Report:
              precision    recall  f1-score   support

      Normal       0.78      0.75      0.77        48
     Missing       0.37      0.41      0.39        17

    accuracy                           0.66        65
   macro avg       0.58      0.58      0.58        65
weighted avg       0.67      0.66      0.67        65



#optimized circular preparation

In [69]:
def prepare_circular(
    circular_id
):

    circular = (
        chunk_df[
            chunk_df["circular_id"]
            == circular_id
        ]
        .copy()
    )

    english = (
        circular[
            circular["language"] == "English"
        ]
        .sort_values("chunk_index")
        .reset_index(drop=True)
    )

    sinhala = (
        circular[
            circular["language"] == "Sinhala"
        ]
        .sort_values("chunk_index")
        .reset_index(drop=True)
    )

    tamil = (
        circular[
            circular["language"] == "Tamil"
        ]
        .sort_values("chunk_index")
        .reset_index(drop=True)
    )


    # Generate embeddings only ONCE per circular
    en_embeddings = model.encode(
        [
            "query: " + text
            for text in english["chunk_text"]
        ],
        normalize_embeddings=True,
        convert_to_tensor=True,
        batch_size=32,
        show_progress_bar=False
    )

    si_embeddings = model.encode(
        [
            "query: " + text
            for text in sinhala["chunk_text"]
        ],
        normalize_embeddings=True,
        convert_to_tensor=True,
        batch_size=32,
        show_progress_bar=False
    )

    ta_embeddings = model.encode(
        [
            "query: " + text
            for text in tamil["chunk_text"]
        ],
        normalize_embeddings=True,
        convert_to_tensor=True,
        batch_size=32,
        show_progress_bar=False
    )


    # Complete English-Sinhala alignment
    complete_similarity = util.cos_sim(
        en_embeddings,
        si_embeddings
    )

    complete_path = dtw_align(
        complete_similarity
    )


    # English-Tamil reference alignment
    en_ta_similarity = util.cos_sim(
        en_embeddings,
        ta_embeddings
    )

    en_ta_path = dtw_align(
        en_ta_similarity
    )

    tamil_scores = get_source_coverage(
        en_ta_path,
        en_ta_similarity,
        len(english)
    )


    return {
        "circular_id": circular_id,
        "english": english,
        "sinhala": sinhala,
        "tamil": tamil,
        "en_embeddings": en_embeddings,
        "si_embeddings": si_embeddings,
        "complete_path": complete_path,
        "tamil_scores": tamil_scores
    }

In [70]:
#fast synthetic case function
def build_synthetic_case_fast(
    prepared,
    remove_positions,
    scenario_name
):

    circular_id = prepared[
        "circular_id"
    ]

    english = prepared[
        "english"
    ]

    sinhala = prepared[
        "sinhala"
    ]

    en_embeddings = prepared[
        "en_embeddings"
    ]

    si_embeddings = prepared[
        "si_embeddings"
    ]

    complete_path = prepared[
        "complete_path"
    ]

    tamil_scores = prepared[
        "tamil_scores"
    ]


    # ----------------------------------------
    # Identify removed Sinhala chunk indices
    # ----------------------------------------

    removed_indices = (
        sinhala
        .iloc[remove_positions][
            "chunk_index"
        ]
        .tolist()
    )


    # ----------------------------------------
    # Find corresponding English ground truth
    # from COMPLETE alignment
    # ----------------------------------------

    affected_english = set()

    for en_pos, si_pos in complete_path:

        if si_pos in remove_positions:

            affected_english.add(
                int(
                    english
                    .iloc[en_pos][
                        "chunk_index"
                    ]
                )
            )


    # ----------------------------------------
    # Remove Sinhala chunks
    # ----------------------------------------

    keep_positions = [
        position
        for position in range(
            len(sinhala)
        )
        if position not in remove_positions
    ]


    sinhala_missing = (
        sinhala
        .iloc[keep_positions]
        .copy()
        .reset_index(drop=True)
    )


    # Reuse already generated Sinhala embeddings
    missing_embeddings = (
        si_embeddings[
            keep_positions
        ]
    )


    # ----------------------------------------
    # Missing-version alignment
    # ----------------------------------------

    missing_similarity = util.cos_sim(
        en_embeddings,
        missing_embeddings
    )

    missing_path = dtw_align(
        missing_similarity
    )


    path_df = create_path_dataframe(
        missing_path,
        missing_similarity,
        english,
        sinhala_missing
    )

    path_df = add_compression_feature(
        path_df
    )


    # ----------------------------------------
    # Sinhala alignment features
    # ----------------------------------------

    sinhala_features = (
        path_df
        .groupby(
            "english_chunk"
        )
        .agg(
            sinhala_similarity=(
                "similarity",
                "max"
            ),

            compression_count=(
                "compression_count",
                "max"
            )
        )
        .reset_index()
    )


    # ----------------------------------------
    # Tamil reference features
    # ----------------------------------------

    tamil_features = pd.DataFrame(
        {
            "english_chunk":
                english[
                    "chunk_index"
                ].tolist(),

            "tamil_similarity":
                tamil_scores
        }
    )


    feature_df = (
        sinhala_features
        .merge(
            tamil_features,
            on="english_chunk",
            how="left"
        )
        .sort_values(
            "english_chunk"
        )
        .reset_index(drop=True)
    )


    # ----------------------------------------
    # Cross-language disagreement
    # ----------------------------------------

    feature_df[
        "cross_language_gap"
    ] = (
        feature_df[
            "tamil_similarity"
        ]
        -
        feature_df[
            "sinhala_similarity"
        ]
    )


    # ----------------------------------------
    # Local semantic anomaly
    # ----------------------------------------

    feature_df[
        "local_median"
    ] = (
        feature_df[
            "sinhala_similarity"
        ]
        .rolling(
            window=5,
            center=True,
            min_periods=1
        )
        .median()
    )

    feature_df[
        "local_drop"
    ] = (
        feature_df[
            "local_median"
        ]
        -
        feature_df[
            "sinhala_similarity"
        ]
    )


    # ----------------------------------------
    # Synthetic ground-truth label
    # ----------------------------------------

    feature_df[
        "synthetic_missing"
    ] = (
        feature_df[
            "english_chunk"
        ]
        .isin(
            affected_english
        )
    )


    feature_df[
        "circular_id"
    ] = circular_id

    feature_df[
        "scenario"
    ] = scenario_name

    feature_df[
        "removed_chunks"
    ] = str(
        removed_indices
    )


    return feature_df

In [71]:
#three missing locations per circular
def create_missing_scenarios(
    sinhala_count
):

    # Three-chunk blocks:
    # early, middle and late sections

    early_start = 1

    middle_start = max(
        1,
        sinhala_count // 2 - 1
    )

    late_start = max(
        1,
        sinhala_count - 4
    )


    scenarios = {
        "early": list(
            range(
                early_start,
                early_start + 3
            )
        ),

        "middle": list(
            range(
                middle_start,
                middle_start + 3
            )
        ),

        "late": list(
            range(
                late_start,
                late_start + 3
            )
        )
    }


    # Remove accidental duplicate scenarios
    unique_scenarios = {}

    seen = set()

    for name, positions in scenarios.items():

        key = tuple(
            positions
        )

        if key not in seen:

            unique_scenarios[
                name
            ] = positions

            seen.add(key)


    return unique_scenarios

In [72]:
# choose 60 NEW circulars
previous_ids = set(
    evaluation_ids
)

fresh_eligible = (
    eligible_circulars[
        ~eligible_circulars.index.isin(
            previous_ids
        )
    ]
)

print(
    "Fresh eligible circulars:",
    len(fresh_eligible)
)


large_evaluation_ids = (
    fresh_eligible
    .sample(
        n=min(
            60,
            len(fresh_eligible)
        ),
        random_state=2026
    )
    .index
    .tolist()
)

print(
    "Selected fresh circulars:",
    len(large_evaluation_ids)
)

Fresh eligible circulars: 124
Selected fresh circulars: 60


In [73]:
#generate large synthetic dataset
large_cases = []

for index, circular_id in enumerate(
    large_evaluation_ids,
    start=1
):

    print(
        f"Processing {index}/"
        f"{len(large_evaluation_ids)}"
        f" - Circular {circular_id}"
    )

    try:

        prepared = prepare_circular(
            circular_id
        )

        scenarios = create_missing_scenarios(
            len(
                prepared["sinhala"]
            )
        )


        for (
            scenario_name,
            positions
        ) in scenarios.items():

            case_df = (
                build_synthetic_case_fast(
                    prepared,
                    positions,
                    scenario_name
                )
            )

            large_cases.append(
                case_df
            )


    except Exception as error:

        print(
            "Failed:",
            circular_id,
            error
        )

Processing 1/60 - Circular 2127
Processing 2/60 - Circular 807
Processing 3/60 - Circular 1807
Processing 4/60 - Circular 2109
Processing 5/60 - Circular 2209
Processing 6/60 - Circular 1621
Processing 7/60 - Circular 1900
Processing 8/60 - Circular 352
Processing 9/60 - Circular 1517
Processing 10/60 - Circular 2097
Processing 11/60 - Circular 1905
Processing 12/60 - Circular 1430
Processing 13/60 - Circular 2132
Processing 14/60 - Circular 2135
Processing 15/60 - Circular 2069
Processing 16/60 - Circular 1812
Processing 17/60 - Circular 634
Processing 18/60 - Circular 2101
Processing 19/60 - Circular 1678
Processing 20/60 - Circular 1719
Processing 21/60 - Circular 1912
Processing 22/60 - Circular 1436
Processing 23/60 - Circular 2210
Processing 24/60 - Circular 176
Processing 25/60 - Circular 1399
Processing 26/60 - Circular 2221
Processing 27/60 - Circular 1783
Processing 28/60 - Circular 309
Processing 29/60 - Circular 1466
Processing 30/60 - Circular 1725
Processing 31/60 - Circu

In [74]:
#combine + validate
large_evaluation_df = pd.concat(
    large_cases,
    ignore_index=True
)


print(
    "\nTotal evaluation rows:",
    len(large_evaluation_df)
)

print(
    "Circulars processed:",
    large_evaluation_df[
        "circular_id"
    ].nunique()
)

print(
    "Synthetic scenarios:",
    large_evaluation_df[
        [
            "circular_id",
            "scenario"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "\nClass distribution:"
)

print(
    large_evaluation_df[
        "synthetic_missing"
    ]
    .value_counts()
)


Total evaluation rows: 2523
Circulars processed: 60
Synthetic scenarios: 180

Class distribution:
synthetic_missing
False    1947
True      576
Name: count, dtype: int64


In [75]:
#save it
output_file = (
    "../data/processed/"
    "synthetic_missing_evaluation.jsonl"
)

large_evaluation_df.to_json(
    output_file,
    orient="records",
    lines=True,
    force_ascii=False
)

print(
    "Saved to:",
    output_file
)

Saved to: ../data/processed/synthetic_missing_evaluation.jsonl


# circular-level train / validation / test split

In [76]:
# Reproducible circular-level split
rng = np.random.default_rng(
    42
)

all_circular_ids = np.array(
    large_evaluation_ids
)

rng.shuffle(
    all_circular_ids
)

train_ids = all_circular_ids[:40]
validation_ids = all_circular_ids[40:50]
test_ids = all_circular_ids[50:60]


train_df = (
    large_evaluation_df[
        large_evaluation_df[
            "circular_id"
        ].isin(train_ids)
    ]
    .copy()
)

validation_df = (
    large_evaluation_df[
        large_evaluation_df[
            "circular_id"
        ].isin(validation_ids)
    ]
    .copy()
)

test_df = (
    large_evaluation_df[
        large_evaluation_df[
            "circular_id"
        ].isin(test_ids)
    ]
    .copy()
)

In [77]:
#verify
print(
    "Training circulars:",
    train_df["circular_id"].nunique()
)

print(
    "Validation circulars:",
    validation_df["circular_id"].nunique()
)

print(
    "Test circulars:",
    test_df["circular_id"].nunique()
)

print(
    "\nTraining rows:",
    len(train_df)
)

print(
    "Validation rows:",
    len(validation_df)
)

print(
    "Test rows:",
    len(test_df)
)

Training circulars: 40
Validation circulars: 10
Test circulars: 10

Training rows: 1698
Validation rows: 417
Test rows: 408


In [78]:
# check class distributions
print("TRAIN")
print(
    train_df[
        "synthetic_missing"
    ].value_counts()
)

print("\nVALIDATION")
print(
    validation_df[
        "synthetic_missing"
    ].value_counts()
)

print("\nTEST")
print(
    test_df[
        "synthetic_missing"
    ].value_counts()
)

TRAIN
synthetic_missing
False    1313
True      385
Name: count, dtype: int64

VALIDATION
synthetic_missing
False    326
True      91
Name: count, dtype: int64

TEST
synthetic_missing
False    308
True     100
Name: count, dtype: int64


In [79]:
#select 40 new circulars
# Circulars already used
used_ids = set(
    large_evaluation_ids
)

# Remaining unused eligible circulars
remaining_circulars = (
    fresh_eligible[
        ~fresh_eligible.index.isin(
            used_ids
        )
    ]
)

print(
    "Remaining unused circulars:",
    len(remaining_circulars)
)

Remaining unused circulars: 64


In [80]:
#select 40
additional_ids = (
    remaining_circulars
    .sample(
        n=40,
        random_state=2027
    )
    .index
    .tolist()
)

print(
    "Additional circulars:",
    len(additional_ids)
)

Additional circulars: 40


In [81]:
#process those 40
additional_cases = []

for index, circular_id in enumerate(
    additional_ids,
    start=1
):

    print(
        f"Processing {index}/40"
        f" - Circular {circular_id}"
    )

    try:

        prepared = prepare_circular(
            circular_id
        )

        scenarios = create_missing_scenarios(
            len(
                prepared["sinhala"]
            )
        )

        for (
            scenario_name,
            positions
        ) in scenarios.items():

            case_df = build_synthetic_case_fast(
                prepared,
                positions,
                scenario_name
            )

            additional_cases.append(
                case_df
            )

    except Exception as error:

        print(
            "Failed:",
            circular_id,
            error
        )

Processing 1/40 - Circular 1373
Processing 2/40 - Circular 2192
Processing 3/40 - Circular 1233
Processing 4/40 - Circular 1126
Processing 5/40 - Circular 1288
Processing 6/40 - Circular 1692
Processing 7/40 - Circular 113
Processing 8/40 - Circular 1501
Processing 9/40 - Circular 369
Processing 10/40 - Circular 2049
Processing 11/40 - Circular 1243
Processing 12/40 - Circular 2111
Processing 13/40 - Circular 1937
Processing 14/40 - Circular 145
Processing 15/40 - Circular 1118
Processing 16/40 - Circular 1720
Processing 17/40 - Circular 341
Processing 18/40 - Circular 390
Processing 19/40 - Circular 1143
Processing 20/40 - Circular 912
Processing 21/40 - Circular 1641
Processing 22/40 - Circular 1612
Processing 23/40 - Circular 1310
Processing 24/40 - Circular 2094
Processing 25/40 - Circular 1857
Processing 26/40 - Circular 1482
Processing 27/40 - Circular 1150
Processing 28/40 - Circular 1944
Processing 29/40 - Circular 244
Processing 30/40 - Circular 2169
Processing 31/40 - Circula

In [82]:
# combine old 60 + new 40
additional_df = pd.concat(
    additional_cases,
    ignore_index=True
)

expanded_evaluation_df = pd.concat(
    [
        large_evaluation_df,
        additional_df
    ],
    ignore_index=True
)

In [83]:
# verify:
print(
    "Total circulars:",
    expanded_evaluation_df[
        "circular_id"
    ].nunique()
)

print(
    "Total rows:",
    len(expanded_evaluation_df)
)

print(
    "Total scenarios:",
    expanded_evaluation_df[
        [
            "circular_id",
            "scenario"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "\nClass distribution:"
)

print(
    expanded_evaluation_df[
        "synthetic_missing"
    ].value_counts()
)

Total circulars: 100
Total rows: 4281
Total scenarios: 300

Class distribution:
synthetic_missing
False    3283
True      998
Name: count, dtype: int64


In [84]:
output_file = (
    "../data/processed/"
    "synthetic_missing_evaluation_100.jsonl"
)

expanded_evaluation_df.to_json(
    output_file,
    orient="records",
    lines=True,
    force_ascii=False
)

print(
    "Saved to:",
    output_file
)

Saved to: ../data/processed/synthetic_missing_evaluation_100.jsonl


In [85]:
#circular-level split
# Get unique circular IDs

all_ids = (
    expanded_evaluation_df[
        "circular_id"
    ]
    .unique()
)


# Reproducible shuffle

rng = np.random.default_rng(
    42
)

rng.shuffle(
    all_ids
)


# 70 / 15 / 15 circular split

train_ids = all_ids[:70]

validation_ids = all_ids[70:85]

test_ids = all_ids[85:100]


train_df = (
    expanded_evaluation_df[
        expanded_evaluation_df[
            "circular_id"
        ].isin(train_ids)
    ]
    .copy()
)

validation_df = (
    expanded_evaluation_df[
        expanded_evaluation_df[
            "circular_id"
        ].isin(validation_ids)
    ]
    .copy()
)

test_df = (
    expanded_evaluation_df[
        expanded_evaluation_df[
            "circular_id"
        ].isin(test_ids)
    ]
    .copy()
)

In [86]:
#verify split
print(
    "Training circulars:",
    train_df["circular_id"].nunique()
)

print(
    "Validation circulars:",
    validation_df["circular_id"].nunique()
)

print(
    "Test circulars:",
    test_df["circular_id"].nunique()
)


print(
    "\nTraining rows:",
    len(train_df)
)

print(
    "Validation rows:",
    len(validation_df)
)

print(
    "Test rows:",
    len(test_df)
)

Training circulars: 70
Validation circulars: 15
Test circulars: 15

Training rows: 2997
Validation rows: 729
Test rows: 555


In [87]:
feature_columns = [
    "sinhala_similarity",
    "tamil_similarity",
    "cross_language_gap",
    "compression_count",
    "local_drop"
]


X_train = train_df[
    feature_columns
].copy()

y_train = (
    train_df[
        "synthetic_missing"
    ]
    .astype(int)
)


X_validation = validation_df[
    feature_columns
].copy()

y_validation = (
    validation_df[
        "synthetic_missing"
    ]
    .astype(int)
)

In [88]:
print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)

X_train: (2997, 5)
X_validation: (729, 5)


In [89]:
#check class balance
print("TRAIN")
print(
    train_df[
        "synthetic_missing"
    ].value_counts()
)

print("\nVALIDATION")
print(
    validation_df[
        "synthetic_missing"
    ].value_counts()
)

print("\nTEST")
print(
    test_df[
        "synthetic_missing"
    ].value_counts()
)

TRAIN
synthetic_missing
False    2294
True      703
Name: count, dtype: int64

VALIDATION
synthetic_missing
False    576
True     153
Name: count, dtype: int64

TEST
synthetic_missing
False    413
True     142
Name: count, dtype: int64


In [90]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.ensemble import HistGradientBoostingClassifier

In [91]:
logistic_model = Pipeline(
    [
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=42
            )
        )
    ]
)


random_forest_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

In [92]:
svm_model = Pipeline(
    [
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            SVC(
                kernel="rbf",
                class_weight="balanced",
                probability=True,
                random_state=42
            )
        )
    ]
)


gradient_boosting_model = (
    HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_depth=6,
        class_weight="balanced",
        random_state=42
    )
)

In [93]:
svm_model.fit(
    X_train,
    y_train
)

gradient_boosting_model.fit(
    X_train,
    y_train
)

print("Additional models trained.")

Additional models trained.


In [94]:
logistic_model.fit(
    X_train,
    y_train
)

random_forest_model.fit(
    X_train,
    y_train
)

models = {
    "Logistic Regression": logistic_model,
    "Random Forest": random_forest_model,
    "SVM (RBF)": svm_model,
    "HistGradientBoosting": gradient_boosting_model

}

validation_results = []

for model_name, current_model in models.items():

    predictions = current_model.predict(
        X_validation
    )

    validation_results.append(
        {
            "model": model_name,

            "accuracy": accuracy_score(
                y_validation,
                predictions
            ),

            "precision": precision_score(
                y_validation,
                predictions,
                zero_division=0
            ),

            "recall": recall_score(
                y_validation,
                predictions,
                zero_division=0
            ),

            "f1_score": f1_score(
                y_validation,
                predictions,
                zero_division=0
            )
        }
    )

validation_results_df = (
    pd.DataFrame(validation_results)
    .sort_values(
        "f1_score",
        ascending=False
    )
)

validation_results_df

,model,accuracy,precision,recall,f1_score
2,SVM (RBF),0.750343,0.440816,0.705882,0.542714
1,Random Forest,0.748971,0.436441,0.673203,0.529563
0,Logistic Regression,0.716049,0.400735,0.712418,0.512941
3,HistGradientBoosting,0.716049,0.392857,0.647059,0.488889


In [95]:
validation_results_df.to_csv(
    "../data/processed/model_validation_results.csv",
    index=False
)

print("Saved validation results.")

Saved validation results.


In [96]:
#SVM validation probabilities
svm_validation_probabilities = (
    svm_model
    .predict_proba(
        X_validation
    )[:, 1]
)

print(
    "Validation probabilities:",
    len(svm_validation_probabilities)
)

Validation probabilities: 729


In [97]:
#threshold search
svm_threshold_results = []

for threshold in np.arange(
    0.30,
    0.71,
    0.05
):

    predictions = (
        svm_validation_probabilities
        >= threshold
    ).astype(int)

    svm_threshold_results.append(
        {
            "threshold": round(
                threshold,
                2
            ),

            "accuracy": accuracy_score(
                y_validation,
                predictions
            ),

            "precision": precision_score(
                y_validation,
                predictions,
                zero_division=0
            ),

            "recall": recall_score(
                y_validation,
                predictions,
                zero_division=0
            ),

            "f1_score": f1_score(
                y_validation,
                predictions,
                zero_division=0
            )
        }
    )


svm_threshold_df = (
    pd.DataFrame(
        svm_threshold_results
    )
    .sort_values(
        "f1_score",
        ascending=False
    )
)

svm_threshold_df

,threshold,accuracy,precision,recall,f1_score
2,0.40,0.786008,0.492537,0.647059,0.559322
3,0.45,0.791495,0.502646,0.620915,0.555556
1,0.35,0.777778,0.478469,0.653595,0.552486
4,0.50,0.806584,0.537500,0.562092,0.549521
0,0.30,0.765432,0.458716,0.653595,0.539084
5,0.55,0.812071,0.556338,0.516340,0.535593
6,0.60,0.802469,0.561644,0.267974,0.362832
7,0.65,0.799726,0.818182,0.058824,0.109756
8,0.70,0.792867,1.000000,0.013072,0.025806


In [98]:
#prepare test data
X_test = test_df[
    feature_columns
].copy()

y_test = (
    test_df[
        "synthetic_missing"
    ]
    .astype(int)
)

print(
    "X_test:",
    X_test.shape
)

X_test: (555, 5)


In [99]:
#final test probabilities
test_probabilities = (
    svm_model
    .predict_proba(
        X_test
    )[:, 1]
)

final_threshold = 0.40

final_predictions = (
    test_probabilities
    >= final_threshold
).astype(int)

In [100]:
#final untouched test evaluation
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


print(
    "Final Test Accuracy:",
    round(
        accuracy_score(
            y_test,
            final_predictions
        ),
        4
    )
)

print(
    "Final Test Precision:",
    round(
        precision_score(
            y_test,
            final_predictions
        ),
        4
    )
)

print(
    "Final Test Recall:",
    round(
        recall_score(
            y_test,
            final_predictions
        ),
        4
    )
)

print(
    "Final Test F1-score:",
    round(
        f1_score(
            y_test,
            final_predictions
        ),
        4
    )
)


print(
    "\nConfusion Matrix:"
)

print(
    confusion_matrix(
        y_test,
        final_predictions
    )
)


print(
    "\nClassification Report:"
)

print(
    classification_report(
        y_test,
        final_predictions,
        target_names=[
            "Normal",
            "Missing"
        ]
    )
)

Final Test Accuracy: 0.7838
Final Test Precision: 0.5705
Final Test Recall: 0.6268
Final Test F1-score: 0.5973

Confusion Matrix:
[[346  67]
 [ 53  89]]

Classification Report:
              precision    recall  f1-score   support

      Normal       0.87      0.84      0.85       413
     Missing       0.57      0.63      0.60       142

    accuracy                           0.78       555
   macro avg       0.72      0.73      0.72       555
weighted avg       0.79      0.78      0.79       555



In [101]:
test_results_df = test_df[
    [
        "circular_id",
        "english_chunk",
        "synthetic_missing"
    ]
].copy()

test_results_df["actual"] = (
    y_test.to_numpy()
)

test_results_df["predicted"] = (
    final_predictions
)

test_results_df["probability"] = (
    test_probabilities
)


output_file = (
    "../data/processed/"
    "final_test_predictions.jsonl"
)

test_results_df.to_json(
    output_file,
    orient="records",
    lines=True,
    force_ascii=False
)

print(
    "Saved to:",
    output_file
)

Saved to: ../data/processed/final_test_predictions.jsonl


In [102]:
import os
import json
import joblib


# Create models folder
os.makedirs(
    "../models",
    exist_ok=True
)


# Save the selected SVM model
joblib.dump(
    svm_model,
    "../models/tricheck_svm.joblib"
)


# Save model configuration
model_config = {
    "model_name": "SVM (RBF)",
    "embedding_model": "intfloat/multilingual-e5-small",
    "threshold": 0.40,
    "features": feature_columns
}


with open(
    "../models/tricheck_config.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        model_config,
        file,
        indent=4
    )


print(
    "Final TriCheck-LK model saved."
)

Final TriCheck-LK model saved.


# Generic Three-Language Missing Information Detection

The previous experiments evaluated Sinhala-side omissions.
This section refactors the approach into a language-independent pipeline
that can evaluate missing information in English, Sinhala, and Tamil.

In [103]:
#Language configuration
language_configs = {
    "English": {
        "anchor": "Sinhala",
        "reference": "Tamil"
    },

    "Sinhala": {
        "anchor": "English",
        "reference": "Tamil"
    },

    "Tamil": {
        "anchor": "English",
        "reference": "Sinhala"
    }
}

language_configs

{'English': {'anchor': 'Sinhala', 'reference': 'Tamil'},
 'Sinhala': {'anchor': 'English', 'reference': 'Tamil'},
 'Tamil': {'anchor': 'English', 'reference': 'Sinhala'}}

In [104]:
#generic circular preparation
def prepare_multilingual_circular(
    circular_id
):

    circular = (
        chunk_df[
            chunk_df["circular_id"]
            == circular_id
        ]
        .copy()
    )

    language_data = {}

    for language in [
        "English",
        "Sinhala",
        "Tamil"
    ]:

        chunks = (
            circular[
                circular["language"]
                == language
            ]
            .sort_values(
                "chunk_index"
            )
            .reset_index(
                drop=True
            )
        )

        embeddings = model.encode(
            [
                "query: " + text
                for text in chunks[
                    "chunk_text"
                ]
            ],
            normalize_embeddings=True,
            convert_to_tensor=True,
            batch_size=32,
            show_progress_bar=False
        )

        language_data[
            language
        ] = {
            "chunks": chunks,
            "embeddings": embeddings
        }

    return language_data

In [105]:
#generic path DataFrame
def create_generic_path_dataframe(
    path,
    similarity_matrix,
    source_chunks,
    target_chunks
):

    rows = []

    for source_pos, target_pos in path:

        score = (
            similarity_matrix[
                source_pos,
                target_pos
            ]
            .item()
        )

        rows.append(
            {
                "source_position":
                    source_pos,

                "target_position":
                    target_pos,

                "source_chunk":
                    int(
                        source_chunks
                        .iloc[source_pos][
                            "chunk_index"
                        ]
                    ),

                "target_chunk":
                    int(
                        target_chunks
                        .iloc[target_pos][
                            "chunk_index"
                        ]
                    ),

                "similarity":
                    score
            }
        )

    return pd.DataFrame(
        rows
    )

In [106]:
#generic compression feature
def add_generic_compression_feature(
    path_df
):

    target_counts = (
        path_df[
            "target_position"
        ]
        .value_counts()
    )

    result = path_df.copy()

    result[
        "compression_count"
    ] = (
        result[
            "target_position"
        ]
        .map(
            target_counts
        )
    )

    return result

In [107]:
#generic synthetic missing case
def build_generic_missing_case(
    prepared,
    target_language,
    remove_positions,
    scenario_name,
    circular_id
):

    config = language_configs[
        target_language
    ]

    anchor_language = (
        config["anchor"]
    )

    reference_language = (
        config["reference"]
    )


    # --------------------------------------------------
    # Get language-specific data
    # --------------------------------------------------

    anchor_chunks = (
        prepared[
            anchor_language
        ]["chunks"]
    )

    target_chunks = (
        prepared[
            target_language
        ]["chunks"]
    )

    reference_chunks = (
        prepared[
            reference_language
        ]["chunks"]
    )


    anchor_embeddings = (
        prepared[
            anchor_language
        ]["embeddings"]
    )

    target_embeddings = (
        prepared[
            target_language
        ]["embeddings"]
    )

    reference_embeddings = (
        prepared[
            reference_language
        ]["embeddings"]
    )


    # --------------------------------------------------
    # Complete anchor ↔ target alignment
    # --------------------------------------------------

    complete_similarity = (
        util.cos_sim(
            anchor_embeddings,
            target_embeddings
        )
    )

    complete_path = dtw_align(
        complete_similarity
    )


    # --------------------------------------------------
    # Anchor ↔ third-language reference
    # --------------------------------------------------

    reference_similarity = (
        util.cos_sim(
            anchor_embeddings,
            reference_embeddings
        )
    )

    reference_path = dtw_align(
        reference_similarity
    )


    reference_scores = (
        get_source_coverage(
            reference_path,
            reference_similarity,
            len(anchor_chunks)
        )
    )


    # --------------------------------------------------
    # Determine removed target chunks
    # --------------------------------------------------

    removed_indices = (
        target_chunks
        .iloc[
            remove_positions
        ][
            "chunk_index"
        ]
        .tolist()
    )


    # --------------------------------------------------
    # Determine ground-truth affected anchor chunks
    # --------------------------------------------------

    affected_source_chunks = set()

    for (
        source_pos,
        target_pos
    ) in complete_path:

        if target_pos in remove_positions:

            affected_source_chunks.add(
                int(
                    anchor_chunks
                    .iloc[source_pos][
                        "chunk_index"
                    ]
                )
            )


    # --------------------------------------------------
    # Remove target-language chunks
    # --------------------------------------------------

    keep_positions = [
        position
        for position in range(
            len(target_chunks)
        )
        if position
        not in remove_positions
    ]


    missing_target_chunks = (
        target_chunks
        .iloc[
            keep_positions
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    missing_target_embeddings = (
        target_embeddings[
            keep_positions
        ]
    )


    # --------------------------------------------------
    # Align anchor with incomplete target
    # --------------------------------------------------

    missing_similarity = (
        util.cos_sim(
            anchor_embeddings,
            missing_target_embeddings
        )
    )

    missing_path = dtw_align(
        missing_similarity
    )


    path_df = (
        create_generic_path_dataframe(
            missing_path,
            missing_similarity,
            anchor_chunks,
            missing_target_chunks
        )
    )

    path_df = (
        add_generic_compression_feature(
            path_df
        )
    )
    # Relative alignment-position difference
    # Measures whether source and target positions
    # become unusually displaced during alignment

    source_max = max(
        len(anchor_chunks) - 1,
        1
    )

    target_max = max(
        len(missing_target_chunks) - 1,
        1
    )

    path_df["position_difference"] = abs(
        (
            path_df["source_position"]
            / source_max
        )
        -
        (
            path_df["target_position"]
            / target_max
        )
    )

    # --------------------------------------------------
    # Target-language alignment features
    # --------------------------------------------------

    target_features = (
        path_df
        .groupby(
            "source_chunk"
        )
        .agg(
            target_similarity=(
                "similarity",
                "max"
            ),

            compression_count=(
                "compression_count",
                "max"
            ),
             position_difference=(
            "position_difference",
            "max"
            )
        )
        .reset_index()
    )


    # --------------------------------------------------
    # Reference-language feature
    # --------------------------------------------------

    reference_features = (
        pd.DataFrame(
            {
                "source_chunk":
                    anchor_chunks[
                        "chunk_index"
                    ].tolist(),

                "reference_similarity":
                    reference_scores
            }
        )
    )


    feature_df = (
        target_features
        .merge(
            reference_features,
            on="source_chunk",
            how="left"
        )
        .sort_values(
            "source_chunk"
        )
        .reset_index(
            drop=True
        )
    )
    feature_df[
    "compression_local_max"
    ] = (
    feature_df[
        "compression_count"
    ]
    .rolling(
        window=3,
        center=True,
        min_periods=1
    )
    .max()
    )

    # --------------------------------------------------
    # Cross-language disagreement
    # --------------------------------------------------

    feature_df[
        "cross_language_gap"
    ] = (
        feature_df[
            "reference_similarity"
        ]
        -
        feature_df[
            "target_similarity"
        ]
    )


    # --------------------------------------------------
    # Local semantic anomaly
    # --------------------------------------------------

    feature_df[
        "local_median"
    ] = (
        feature_df[
            "target_similarity"
        ]
        .rolling(
            window=5,
            center=True,
            min_periods=1
        )
        .median()
    )


    feature_df[
        "local_drop"
    ] = (
        feature_df[
            "local_median"
        ]
        -
        feature_df[
            "target_similarity"
        ]
    )


    # --------------------------------------------------
    # Ground-truth synthetic label
    # --------------------------------------------------

    feature_df[
        "synthetic_missing"
    ] = (
        feature_df[
            "source_chunk"
        ]
        .isin(
            affected_source_chunks
        )
    )


    # --------------------------------------------------
    # Metadata
    # --------------------------------------------------

    feature_df[
        "circular_id"
    ] = circular_id

    feature_df[
        "target_language"
    ] = target_language

    feature_df[
        "anchor_language"
    ] = anchor_language

    feature_df[
        "reference_language"
    ] = reference_language

    feature_df[
        "scenario"
    ] = scenario_name

    feature_df[
        "removed_chunks"
    ] = str(
        removed_indices
    )


    return feature_df

In [108]:
#Test before running 100 circulars
test_circular_id = 2221

prepared_test = (
    prepare_multilingual_circular(
        test_circular_id
    )
)


for target_language in [
    "English",
    "Sinhala",
    "Tamil"
]:

    target_count = len(
        prepared_test[
            target_language
        ]["chunks"]
    )

    middle = (
        target_count // 2
    )

    remove_positions = [
        middle - 1,
        middle,
        middle + 1
    ]


    test_result = (
        build_generic_missing_case(
            prepared=prepared_test,
            target_language=target_language,
            remove_positions=remove_positions,
            scenario_name="middle",
            circular_id=test_circular_id
        )
    )


    print(
        "\nTarget:",
        target_language
    )

    print(
        "Rows:",
        len(test_result)
    )

    print(
        "Missing labels:",
        test_result[
            "synthetic_missing"
        ].sum()
    )

    print(
        "Removed:",
        test_result[
            "removed_chunks"
        ].iloc[0]
    )


Target: English
Rows: 25
Missing labels: 3
Removed: [11, 12, 13]

Target: Sinhala
Rows: 25
Missing labels: 3
Removed: [11, 12, 13]

Target: Tamil
Rows: 25
Missing labels: 3
Removed: [13, 14, 15]


In [109]:
generic_evaluation_ids = (
    expanded_evaluation_df[
        "circular_id"
    ]
    .drop_duplicates()
    .tolist()
)

print(
    "Evaluation circulars:",
    len(generic_evaluation_ids)
)

print(
    generic_evaluation_ids[:10]
)

Evaluation circulars: 100
[2127, 807, 1807, 2109, 2209, 1621, 1900, 352, 1517, 2097]


In [110]:
#generate three-language evaluation dataset
generic_cases = []

target_languages = [
    "English",
    "Sinhala",
    "Tamil"
]


for index, circular_id in enumerate(
    generic_evaluation_ids,
    start=1
):

    print(
        f"Processing {index}/"
        f"{len(generic_evaluation_ids)}"
        f" - Circular {circular_id}"
    )

    try:

        # Generate EN / SI / TA embeddings
        # only once for this circular
        prepared = (
            prepare_multilingual_circular(
                circular_id
            )
        )


        # --------------------------------------
        # Test missing information
        # in each of the three languages
        # --------------------------------------

        for target_language in target_languages:

            target_count = len(
                prepared[
                    target_language
                ]["chunks"]
            )


            scenarios = (
                create_missing_scenarios(
                    target_count
                )
            )


            for (
                scenario_name,
                positions
            ) in scenarios.items():

                case_df = (
                    build_generic_missing_case(
                        prepared=prepared,
                        target_language=target_language,
                        remove_positions=positions,
                        scenario_name=scenario_name,
                        circular_id=circular_id
                    )
                )

                generic_cases.append(
                    case_df
                )


    except Exception as error:

        print(
            "Failed:",
            circular_id,
            error
        )

Processing 1/100 - Circular 2127
Processing 2/100 - Circular 807
Processing 3/100 - Circular 1807
Processing 4/100 - Circular 2109
Processing 5/100 - Circular 2209
Processing 6/100 - Circular 1621
Processing 7/100 - Circular 1900
Processing 8/100 - Circular 352
Processing 9/100 - Circular 1517
Processing 10/100 - Circular 2097
Processing 11/100 - Circular 1905
Processing 12/100 - Circular 1430
Processing 13/100 - Circular 2132
Processing 14/100 - Circular 2135
Processing 15/100 - Circular 2069
Processing 16/100 - Circular 1812
Processing 17/100 - Circular 634
Processing 18/100 - Circular 2101
Processing 19/100 - Circular 1678
Processing 20/100 - Circular 1719
Processing 21/100 - Circular 1912
Processing 22/100 - Circular 1436
Processing 23/100 - Circular 2210
Processing 24/100 - Circular 176
Processing 25/100 - Circular 1399
Processing 26/100 - Circular 2221
Processing 27/100 - Circular 1783
Processing 28/100 - Circular 309
Processing 29/100 - Circular 1466
Processing 30/100 - Circular

In [111]:
#combine all results
generic_evaluation_df = pd.concat(
    generic_cases,
    ignore_index=True
)

In [112]:
#validation
print(
    "Total evaluation rows:",
    len(generic_evaluation_df)
)

print(
    "Circulars:",
    generic_evaluation_df[
        "circular_id"
    ].nunique()
)

print(
    "Target languages:",
    generic_evaluation_df[
        "target_language"
    ].nunique()
)

print(
    "Synthetic scenarios:",
    generic_evaluation_df[
        [
            "circular_id",
            "target_language",
            "scenario"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

Total evaluation rows: 12480
Circulars: 100
Target languages: 3
Synthetic scenarios: 900


In [113]:
# Check number of evaluation rows by target language

print(
    generic_evaluation_df[
        "target_language"
    ].value_counts()
)


print(
    "\nOverall class distribution:"
)

print(
    generic_evaluation_df[
        "synthetic_missing"
    ].value_counts()
)


print(
    "\nClass distribution by target language:"
)

print(
    pd.crosstab(
        generic_evaluation_df[
            "target_language"
        ],
        generic_evaluation_df[
            "synthetic_missing"
        ]
    )
)

target_language
Sinhala    4281
Tamil      4281
English    3918
Name: count, dtype: int64

Overall class distribution:
synthetic_missing
False    9697
True     2783
Name: count, dtype: int64

Class distribution by target language:
synthetic_missing  False  True 
target_language                
English             3046    872
Sinhala             3283    998
Tamil               3368    913


In [114]:
output_file = (
    "../data/processed/"
    "synthetic_multilingual_missing_evaluation_100.jsonl"
)

generic_evaluation_df.to_json(
    output_file,
    orient="records",
    lines=True,
    force_ascii=False
)

print(
    "Saved to:",
    output_file
)

Saved to: ../data/processed/synthetic_multilingual_missing_evaluation_100.jsonl


## Generic Multilingual Missing-Information Classifier

A single classifier is trained to identify potential missing information
regardless of whether the target language is English, Sinhala, or Tamil.

The train, validation, and test sets are separated by circular ID to avoid
document-level data leakage.

In [115]:
#reuse the same 70 / 15 / 15 circular split
generic_train_df = (
    generic_evaluation_df[
        generic_evaluation_df["circular_id"]
        .isin(train_ids)
    ]
    .copy()
)

generic_validation_df = (
    generic_evaluation_df[
        generic_evaluation_df["circular_id"]
        .isin(validation_ids)
    ]
    .copy()
)

generic_test_df = (
    generic_evaluation_df[
        generic_evaluation_df["circular_id"]
        .isin(test_ids)
    ]
    .copy()
)


print(
    "Training circulars:",
    generic_train_df["circular_id"].nunique()
)

print(
    "Validation circulars:",
    generic_validation_df["circular_id"].nunique()
)

print(
    "Test circulars:",
    generic_test_df["circular_id"].nunique()
)

print(
    "\nTraining rows:",
    len(generic_train_df)
)

print(
    "Validation rows:",
    len(generic_validation_df)
)

print(
    "Test rows:",
    len(generic_test_df)
)

Training circulars: 70
Validation circulars: 15
Test circulars: 15

Training rows: 8742
Validation rows: 2097
Test rows: 1641


In [116]:
#prepare generic features
generic_feature_columns = [
    "target_similarity",
    "reference_similarity",
    "cross_language_gap",
    "compression_count",
    "local_drop",
    "target_language"
]


generic_features = (
    pd.get_dummies(
        generic_evaluation_df[
            generic_feature_columns
        ],
        columns=[
            "target_language"
        ],
        dtype=int
    )
)

generic_features.head()

,target_similarity,reference_similarity,cross_language_gap,compression_count,local_drop,target_language_English,target_language_Sinhala,target_language_Tamil
0,0.881664,0.921373,0.039708,4,-0.030532,1,0,0
1,0.848158,0.908285,0.060127,4,0.007379,1,0,0
2,0.851133,0.911125,0.059993,4,0.000000,1,0,0
3,0.859942,0.934442,0.074500,4,-0.008809,1,0,0
4,0.817416,0.895879,0.078463,1,0.040013,1,0,0


In [117]:
#create Train / Validation / Test X and y
X_generic_train = (
    generic_features.loc[
        generic_train_df.index
    ]
)

y_generic_train = (
    generic_train_df[
        "synthetic_missing"
    ]
    .astype(int)
)


X_generic_validation = (
    generic_features.loc[
        generic_validation_df.index
    ]
)

y_generic_validation = (
    generic_validation_df[
        "synthetic_missing"
    ]
    .astype(int)
)


X_generic_test = (
    generic_features.loc[
        generic_test_df.index
    ]
)

y_generic_test = (
    generic_test_df[
        "synthetic_missing"
    ]
    .astype(int)
)


print(
    "Train:",
    X_generic_train.shape
)

print(
    "Validation:",
    X_generic_validation.shape
)

print(
    "Test:",
    X_generic_test.shape
)

Train: (8742, 8)
Validation: (2097, 8)
Test: (1641, 8)


#generic 3-language model comparison

In [118]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier
)
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [119]:
#define 4 models
generic_logistic_model = Pipeline(
    [
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=42
            )
        )
    ]
)


generic_random_forest_model = (
    RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
)


generic_svm_model = Pipeline(
    [
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            SVC(
                kernel="rbf",
                class_weight="balanced",
                probability=True,
                random_state=42
            )
        )
    ]
)


generic_gradient_model = (
    HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_depth=6,
        class_weight="balanced",
        random_state=42
    )
)

In [120]:
#train
generic_logistic_model.fit(
    X_generic_train,
    y_generic_train
)

generic_random_forest_model.fit(
    X_generic_train,
    y_generic_train
)

generic_svm_model.fit(
    X_generic_train,
    y_generic_train
)

generic_gradient_model.fit(
    X_generic_train,
    y_generic_train
)

print(
    "All generic multilingual models trained."
)

All generic multilingual models trained.


#validation comparison

In [121]:
generic_models = {
    "Logistic Regression":
        generic_logistic_model,

    "Random Forest":
        generic_random_forest_model,

    "SVM (RBF)":
        generic_svm_model,

    "HistGradientBoosting":
        generic_gradient_model
}


generic_validation_results = []


for model_name, current_model in generic_models.items():

    predictions = current_model.predict(
        X_generic_validation
    )

    generic_validation_results.append(
        {
            "model": model_name,

            "accuracy": accuracy_score(
                y_generic_validation,
                predictions
            ),

            "precision": precision_score(
                y_generic_validation,
                predictions,
                zero_division=0
            ),

            "recall": recall_score(
                y_generic_validation,
                predictions,
                zero_division=0
            ),

            "f1_score": f1_score(
                y_generic_validation,
                predictions,
                zero_division=0
            )
        }
    )


generic_validation_results_df = (
    pd.DataFrame(
        generic_validation_results
    )
    .sort_values(
        "f1_score",
        ascending=False
    )
)

generic_validation_results_df

,model,accuracy,precision,recall,f1_score
1,Random Forest,0.740105,0.389545,0.557971,0.458788
2,SVM (RBF),0.735813,0.383721,0.557971,0.454724
3,HistGradientBoosting,0.718169,0.365706,0.582126,0.449208
0,Logistic Regression,0.691464,0.345695,0.630435,0.446536


#validation threshold tuning

In [122]:
# Get validation probabilities
generic_rf_validation_probabilities = (
    generic_random_forest_model
    .predict_proba(
        X_generic_validation
    )[:, 1]
)

print(
    "Validation probabilities:",
    len(generic_rf_validation_probabilities)
)

Validation probabilities: 2097


In [123]:
generic_threshold_results = []

for threshold in np.arange(
    0.20,
    0.71,
    0.05
):

    predictions = (
        generic_rf_validation_probabilities
        >= threshold
    ).astype(int)

    generic_threshold_results.append(
        {
            "threshold": round(
                threshold,
                2
            ),

            "accuracy": accuracy_score(
                y_generic_validation,
                predictions
            ),

            "precision": precision_score(
                y_generic_validation,
                predictions,
                zero_division=0
            ),

            "recall": recall_score(
                y_generic_validation,
                predictions,
                zero_division=0
            ),

            "f1_score": f1_score(
                y_generic_validation,
                predictions,
                zero_division=0
            )
        }
    )


generic_threshold_df = (
    pd.DataFrame(
        generic_threshold_results
    )
    .sort_values(
        "f1_score",
        ascending=False
    )
)

generic_threshold_df

,threshold,accuracy,precision,recall,f1_score
9,0.65,0.818312,0.552381,0.420290,0.477366
10,0.70,0.824988,0.581882,0.403382,0.476462
8,0.60,0.799237,0.490862,0.454106,0.471769
6,0.50,0.740105,0.389545,0.557971,0.458788
5,0.45,0.680496,0.344282,0.683575,0.457929
7,0.55,0.768717,0.425263,0.487923,0.454443
4,0.40,0.592752,0.294393,0.760870,0.424528
3,0.35,0.525990,0.273083,0.842995,0.412530
2,0.30,0.428231,0.245295,0.913043,0.386701
1,0.25,0.349547,0.227950,0.961353,0.368519


முதலில் validation set-ல் English / Sinhala / Tamil தனித்தனியாக performance check செய்ய வேண்டும்.

In [124]:
generic_validation_predictions = (
    generic_rf_validation_probabilities
    >= 0.65
).astype(int)

validation_language_df = (
    generic_validation_df.copy()
)

validation_language_df[
    "prediction"
] = generic_validation_predictions

In [125]:
#language-wise metrics
language_results = []

for language in [
    "English",
    "Sinhala",
    "Tamil"
]:

    language_data = (
        validation_language_df[
            validation_language_df[
                "target_language"
            ]
            == language
        ]
    )

    y_true = (
        language_data[
            "synthetic_missing"
        ]
        .astype(int)
    )

    y_pred = (
        language_data[
            "prediction"
        ]
    )

    language_results.append(
        {
            "language": language,

            "accuracy": accuracy_score(
                y_true,
                y_pred
            ),

            "precision": precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),

            "recall": recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),

            "f1_score": f1_score(
                y_true,
                y_pred,
                zero_division=0
            )
        }
    )


language_validation_results = (
    pd.DataFrame(
        language_results
    )
)

language_validation_results

,language,accuracy,precision,recall,f1_score
0,English,0.810642,0.562500,0.144000,0.229299
1,Sinhala,0.802469,0.525424,0.607843,0.563636
2,Tamil,0.840878,0.594340,0.463235,0.520661


In [126]:
validation_threshold_df = (
    generic_validation_df.copy()
)

validation_threshold_df[
    "missing_probability"
] = generic_rf_validation_probabilities

In [127]:
language_threshold_results = []

for language in [
    "English",
    "Sinhala",
    "Tamil"
]:

    language_data = (
        validation_threshold_df[
            validation_threshold_df[
                "target_language"
            ]
            == language
        ]
    )

    y_true = (
        language_data[
            "synthetic_missing"
        ]
        .astype(int)
        .to_numpy()
    )

    probabilities = (
        language_data[
            "missing_probability"
        ]
        .to_numpy()
    )

    for threshold in np.arange(
        0.20,
        0.76,
        0.05
    ):

        y_pred = (
            probabilities
            >= threshold
        ).astype(int)

        language_threshold_results.append(
            {
                "language": language,
                "threshold": round(
                    threshold,
                    2
                ),

                "precision": precision_score(
                    y_true,
                    y_pred,
                    zero_division=0
                ),

                "recall": recall_score(
                    y_true,
                    y_pred,
                    zero_division=0
                ),

                "f1_score": f1_score(
                    y_true,
                    y_pred,
                    zero_division=0
                )
            }
        )


language_threshold_df = pd.DataFrame(
    language_threshold_results
)

In [128]:
#best threshold for each language
best_language_thresholds = (
    language_threshold_df
    .sort_values(
        [
            "language",
            "f1_score"
        ],
        ascending=[
            True,
            False
        ]
    )
    .groupby(
        "language",
        as_index=False
    )
    .first()
)

best_language_thresholds

,language,threshold,precision,recall,f1_score
0,English,0.45,0.263889,0.608000,0.368039
1,Sinhala,0.65,0.525424,0.607843,0.563636
2,Tamil,0.70,0.639175,0.455882,0.532189


In [129]:
language_thresholds = {
    "English": 0.45,
    "Sinhala": 0.65,
    "Tamil": 0.70
}

In [130]:
validation_final_df = (
    generic_validation_df.copy()
)

validation_final_df[
    "missing_probability"
] = generic_rf_validation_probabilities


validation_final_df[
    "threshold"
] = (
    validation_final_df[
        "target_language"
    ]
    .map(
        language_thresholds
    )
)


validation_final_df[
    "prediction"
] = (
    validation_final_df[
        "missing_probability"
    ]
    >=
    validation_final_df[
        "threshold"
    ]
).astype(int)

In [131]:
#Overall validation performance
y_validation_final = (
    validation_final_df[
        "synthetic_missing"
    ]
    .astype(int)
)

pred_validation_final = (
    validation_final_df[
        "prediction"
    ]
)


print(
    "Accuracy:",
    round(
        accuracy_score(
            y_validation_final,
            pred_validation_final
        ),
        4
    )
)

print(
    "Precision:",
    round(
        precision_score(
            y_validation_final,
            pred_validation_final
        ),
        4
    )
)

print(
    "Recall:",
    round(
        recall_score(
            y_validation_final,
            pred_validation_final
        ),
        4
    )
)

print(
    "F1-score:",
    round(
        f1_score(
            y_validation_final,
            pred_validation_final
        ),
        4
    )
)

Accuracy: 0.7549
Precision: 0.411
Recall: 0.558
F1-score: 0.4734


In [132]:
final_language_validation = []

for language in [
    "English",
    "Sinhala",
    "Tamil"
]:

    language_df = (
        validation_final_df[
            validation_final_df[
                "target_language"
            ]
            == language
        ]
    )

    y_true = (
        language_df[
            "synthetic_missing"
        ]
        .astype(int)
    )

    y_pred = (
        language_df[
            "prediction"
        ]
    )

    final_language_validation.append(
        {
            "language": language,

            "precision": precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),

            "recall": recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),

            "f1_score": f1_score(
                y_true,
                y_pred,
                zero_division=0
            )
        }
    )


pd.DataFrame(
    final_language_validation
)

,language,precision,recall,f1_score
0,English,0.263889,0.608000,0.368039
1,Sinhala,0.525424,0.607843,0.563636
2,Tamil,0.639175,0.455882,0.532189


## Language-Specific Missing-Information Detectors

Separate classifiers are evaluated for English, Sinhala and Tamil
to account for language-specific alignment and extraction characteristics.

In [134]:
#core features
language_feature_columns = [
    "target_similarity",
    "reference_similarity",
    "cross_language_gap",
    "compression_count",
    "local_drop"
]

In [135]:
#model factory
def create_candidate_models():

    return {
        "Logistic Regression": Pipeline(
            [
                (
                    "scaler",
                    StandardScaler()
                ),
                (
                    "classifier",
                    LogisticRegression(
                        class_weight="balanced",
                        max_iter=1000,
                        random_state=42
                    )
                )
            ]
        ),

        "Random Forest": RandomForestClassifier(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=3,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),

        "SVM (RBF)": Pipeline(
            [
                (
                    "scaler",
                    StandardScaler()
                ),
                (
                    "classifier",
                    SVC(
                        kernel="rbf",
                        class_weight="balanced",
                        probability=True,
                        random_state=42
                    )
                )
            ]
        ),

        "HistGradientBoosting":
            HistGradientBoostingClassifier(
                max_iter=200,
                learning_rate=0.05,
                max_depth=6,
                class_weight="balanced",
                random_state=42
            )
    }

In [136]:
# validation comparison for all 3 languages 
language_model_results = []

trained_language_models = {}


for language in [
    "English",
    "Sinhala",
    "Tamil"
]:

    print(
        "\nTraining:",
        language
    )


    # -------------------------------------
    # Training data
    # -------------------------------------

    train_language_df = (
        generic_train_df[
            generic_train_df[
                "target_language"
            ]
            == language
        ]
    )


    validation_language_df = (
        generic_validation_df[
            generic_validation_df[
                "target_language"
            ]
            == language
        ]
    )


    X_train_language = (
        train_language_df[
            language_feature_columns
        ]
    )

    y_train_language = (
        train_language_df[
            "synthetic_missing"
        ]
        .astype(int)
    )


    X_validation_language = (
        validation_language_df[
            language_feature_columns
        ]
    )

    y_validation_language = (
        validation_language_df[
            "synthetic_missing"
        ]
        .astype(int)
    )


    # -------------------------------------
    # Train candidate models
    # -------------------------------------

    models = create_candidate_models()

    trained_language_models[
        language
    ] = {}


    for model_name, current_model in models.items():

        current_model.fit(
            X_train_language,
            y_train_language
        )

        predictions = (
            current_model.predict(
                X_validation_language
            )
        )


        language_model_results.append(
            {
                "language": language,
                "model": model_name,

                "accuracy": accuracy_score(
                    y_validation_language,
                    predictions
                ),

                "precision": precision_score(
                    y_validation_language,
                    predictions,
                    zero_division=0
                ),

                "recall": recall_score(
                    y_validation_language,
                    predictions,
                    zero_division=0
                ),

                "f1_score": f1_score(
                    y_validation_language,
                    predictions,
                    zero_division=0
                )
            }
        )


        trained_language_models[
            language
        ][
            model_name
        ] = current_model


Training: English

Training: Sinhala

Training: Tamil


In [137]:
language_model_results_df = (
    pd.DataFrame(
        language_model_results
    )
    .sort_values(
        [
            "language",
            "f1_score"
        ],
        ascending=[
            True,
            False
        ]
    )
)

language_model_results_df

,language,model,accuracy,precision,recall,f1_score
0,English,Logistic Regression,0.622848,0.266129,0.528000,0.353887
2,English,SVM (RBF),0.713615,0.298611,0.344000,0.319703
3,English,HistGradientBoosting,0.666667,0.265957,0.400000,0.319489
1,English,Random Forest,0.737089,0.306306,0.272000,0.288136
6,Sinhala,SVM (RBF),0.750343,0.440816,0.705882,0.542714
5,Sinhala,Random Forest,0.748971,0.436441,0.673203,0.529563
4,Sinhala,Logistic Regression,0.716049,0.400735,0.712418,0.512941
7,Sinhala,HistGradientBoosting,0.716049,0.392857,0.647059,0.488889
9,Tamil,Random Forest,0.768176,0.415385,0.595588,0.489426
8,Tamil,Logistic Regression,0.780521,0.431818,0.558824,0.487179


#select best model for each language

In [138]:
selected_language_models = {
    "English": trained_language_models[
        "English"
    ][
        "Logistic Regression"
    ],

    "Sinhala": trained_language_models[
        "Sinhala"
    ][
        "SVM (RBF)"
    ],

    "Tamil": trained_language_models[
        "Tamil"
    ][
        "Random Forest"
    ]
}

In [139]:
#validation threshold tuning
language_threshold_results = []

for language in [
    "English",
    "Sinhala",
    "Tamil"
]:

    validation_language_df = (
        generic_validation_df[
            generic_validation_df[
                "target_language"
            ]
            == language
        ]
        .copy()
    )


    X_val_language = (
        validation_language_df[
            language_feature_columns
        ]
    )

    y_val_language = (
        validation_language_df[
            "synthetic_missing"
        ]
        .astype(int)
        .to_numpy()
    )


    current_model = (
        selected_language_models[
            language
        ]
    )


    probabilities = (
        current_model
        .predict_proba(
            X_val_language
        )[:, 1]
    )


    for threshold in np.arange(
        0.20,
        0.81,
        0.05
    ):

        predictions = (
            probabilities
            >= threshold
        ).astype(int)


        language_threshold_results.append(
            {
                "language": language,

                "threshold": round(
                    threshold,
                    2
                ),

                "precision": precision_score(
                    y_val_language,
                    predictions,
                    zero_division=0
                ),

                "recall": recall_score(
                    y_val_language,
                    predictions,
                    zero_division=0
                ),

                "f1_score": f1_score(
                    y_val_language,
                    predictions,
                    zero_division=0
                )
            }
        )

In [140]:
#get best threshold per language
language_threshold_results_df = (
    pd.DataFrame(
        language_threshold_results
    )
)


best_language_configuration = (
    language_threshold_results_df
    .sort_values(
        [
            "language",
            "f1_score"
        ],
        ascending=[
            True,
            False
        ]
    )
    .groupby(
        "language",
        as_index=False
    )
    .first()
)


best_language_configuration

,language,threshold,precision,recall,f1_score
0,English,0.3,0.218579,0.960000,0.356083
1,Sinhala,0.4,0.492537,0.647059,0.559322
2,Tamil,0.5,0.415385,0.595588,0.489426


In [141]:
#ADD final configuration
final_language_models = {
    "English": selected_language_models["English"],
    "Sinhala": selected_language_models["Sinhala"],
    "Tamil": selected_language_models["Tamil"]
}


final_language_thresholds = {
    "English": 0.30,
    "Sinhala": 0.40,
    "Tamil": 0.50
}

In [142]:
#final language-wise test evaluation
final_test_results = []

test_predictions_list = []


for language in [
    "English",
    "Sinhala",
    "Tamil"
]:

    # --------------------------------------
    # Select unseen test data for language
    # --------------------------------------

    language_test_df = (
        generic_test_df[
            generic_test_df[
                "target_language"
            ]
            == language
        ]
        .copy()
    )


    X_test_language = (
        language_test_df[
            language_feature_columns
        ]
    )

    y_test_language = (
        language_test_df[
            "synthetic_missing"
        ]
        .astype(int)
        .to_numpy()
    )


    # --------------------------------------
    # Get selected model and threshold
    # --------------------------------------

    current_model = (
        final_language_models[
            language
        ]
    )

    threshold = (
        final_language_thresholds[
            language
        ]
    )


    # --------------------------------------
    # Predict probabilities
    # --------------------------------------

    probabilities = (
        current_model
        .predict_proba(
            X_test_language
        )[:, 1]
    )


    predictions = (
        probabilities
        >= threshold
    ).astype(int)


    # --------------------------------------
    # Save metrics
    # --------------------------------------

    final_test_results.append(
        {
            "language": language,

            "threshold": threshold,

            "accuracy": accuracy_score(
                y_test_language,
                predictions
            ),

            "precision": precision_score(
                y_test_language,
                predictions,
                zero_division=0
            ),

            "recall": recall_score(
                y_test_language,
                predictions,
                zero_division=0
            ),

            "f1_score": f1_score(
                y_test_language,
                predictions,
                zero_division=0
            )
        }
    )


    # Save predictions for later evaluation/app
    language_test_df[
        "missing_probability"
    ] = probabilities

    language_test_df[
        "prediction"
    ] = predictions

    test_predictions_list.append(
        language_test_df
    )

In [143]:
from sklearn.metrics import confusion_matrix

# Combine saved language-wise test predictions
all_test_predictions = pd.concat(
    test_predictions_list,
    ignore_index=True
)


for language in [
    "English",
    "Sinhala",
    "Tamil"
]:

    language_df = all_test_predictions[
        all_test_predictions[
            "target_language"
        ] == language
    ]

    y_true = (
        language_df[
            "synthetic_missing"
        ]
        .astype(int)
    )

    y_pred = (
        language_df[
            "prediction"
        ]
        .astype(int)
    )

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    print(
        f"\n{language} Confusion Matrix:"
    )

    print(cm)


English Confusion Matrix:
[[ 78 316]
 [  2 135]]

Sinhala Confusion Matrix:
[[346  67]
 [ 53  89]]

Tamil Confusion Matrix:
[[354  62]
 [ 42  97]]


In [144]:
#show final results
final_multilingual_results_df = (
    pd.DataFrame(
        final_test_results
    )
)

final_multilingual_results_df

,language,threshold,accuracy,precision,recall,f1_score
0,English,0.3,0.401130,0.299335,0.985401,0.459184
1,Sinhala,0.4,0.783784,0.570513,0.626761,0.597315
2,Tamil,0.5,0.812613,0.610063,0.697842,0.651007


In [145]:
#combined overall performance
final_multilingual_test_df = pd.concat(
    test_predictions_list,
    ignore_index=True
)


y_true_all = (
    final_multilingual_test_df[
        "synthetic_missing"
    ]
    .astype(int)
)

y_pred_all = (
    final_multilingual_test_df[
        "prediction"
    ]
)


print(
    "Overall Accuracy:",
    round(
        accuracy_score(
            y_true_all,
            y_pred_all
        ),
        4
    )
)

print(
    "Overall Precision:",
    round(
        precision_score(
            y_true_all,
            y_pred_all
        ),
        4
    )
)

print(
    "Overall Recall:",
    round(
        recall_score(
            y_true_all,
            y_pred_all
        ),
        4
    )
)

print(
    "Overall F1-score:",
    round(
        f1_score(
            y_true_all,
            y_pred_all
        ),
        4
    )
)

Overall Accuracy: 0.6697
Overall Precision: 0.4191
Overall Recall: 0.7679
Overall F1-score: 0.5422


In [146]:
from sklearn.metrics import confusion_matrix
print([
    name
    for name in globals()
    if "tamil" in name.lower()
])
# print(
#     confusion_matrix(
#         y_tamil_test,
#         tamil_test_predictions
#     )
# )

['tamil_chunks', 'tamil_embeddings', 'tamil_features']


#### English Detector Improvement

In [147]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)

In [148]:
english_train_df = (
    generic_train_df[
        generic_train_df["target_language"]
        == "English"
    ]
    .copy()
)

english_validation_df = (
    generic_validation_df[
        generic_validation_df["target_language"]
        == "English"
    ]
    .copy()
)


X_en_train = english_train_df[
    language_feature_columns
]

y_en_train = (
    english_train_df[
        "synthetic_missing"
    ]
    .astype(int)
)


X_en_validation = english_validation_df[
    language_feature_columns
]

y_en_validation = (
    english_validation_df[
        "synthetic_missing"
    ]
    .astype(int)
)

In [149]:
#tune class weight + threshold
english_tuning_results = []

class_weight_options = {
    "none": None,
    "mild": {0: 1.0, 1: 1.5},
    "medium": {0: 1.0, 1: 2.0},
    "balanced": "balanced"
}


for weight_name, class_weight in class_weight_options.items():

    current_model = Pipeline(
        [
            (
                "scaler",
                StandardScaler()
            ),
            (
                "classifier",
                LogisticRegression(
                    class_weight=class_weight,
                    max_iter=1000,
                    random_state=42
                )
            )
        ]
    )


    current_model.fit(
        X_en_train,
        y_en_train
    )


    probabilities = (
        current_model
        .predict_proba(
            X_en_validation
        )[:, 1]
    )


    for threshold in np.arange(
        0.30,
        0.76,
        0.05
    ):

        predictions = (
            probabilities
            >= threshold
        ).astype(int)


        recall = recall_score(
            y_en_validation,
            predictions,
            zero_division=0
        )


        english_tuning_results.append(
            {
                "class_weight":
                    weight_name,

                "threshold":
                    round(
                        threshold,
                        2
                    ),

                "accuracy":
                    accuracy_score(
                        y_en_validation,
                        predictions
                    ),

                "balanced_accuracy":
                    balanced_accuracy_score(
                        y_en_validation,
                        predictions
                    ),

                "precision":
                    precision_score(
                        y_en_validation,
                        predictions,
                        zero_division=0
                    ),

                "recall":
                    recall,

                "f1_score":
                    f1_score(
                        y_en_validation,
                        predictions,
                        zero_division=0
                    )
            }
        )


english_tuning_df = pd.DataFrame(
    english_tuning_results
)

In [150]:
#find better configurations
better_english_configs = (
    english_tuning_df[
        english_tuning_df[
            "recall"
        ] >= 0.60
    ]
    .sort_values(
        [
            "accuracy",
            "f1_score"
        ],
        ascending=False
    )
)

better_english_configs.head(10)

,class_weight,threshold,accuracy,balanced_accuracy,precision,recall,f1_score
33,balanced,0.45,0.516432,0.578327,0.240113,0.680,0.354906
20,medium,0.30,0.466354,0.574444,0.232673,0.752,0.355388
32,balanced,0.40,0.427230,0.562233,0.224256,0.784,0.348754
31,balanced,0.35,0.366197,0.557595,0.218876,0.872,0.349920
30,balanced,0.30,0.320814,0.562685,0.218579,0.960,0.356083


In [151]:
print(
    generic_evaluation_df.columns.tolist()
)

['source_chunk', 'target_similarity', 'compression_count', 'position_difference', 'reference_similarity', 'compression_local_max', 'cross_language_gap', 'local_median', 'local_drop', 'synthetic_missing', 'circular_id', 'target_language', 'anchor_language', 'reference_language', 'scenario', 'removed_chunks']


In [152]:
improved_feature_columns = [
    "target_similarity",
    "reference_similarity",
    "cross_language_gap",
    "compression_count",
    "local_drop",
    "position_difference",
    "compression_local_max"
]

In [153]:
english_train_df = (
    generic_evaluation_df[
        generic_evaluation_df["circular_id"].isin(
            train_ids
        )
        &
        (
            generic_evaluation_df[
                "target_language"
            ] == "English"
        )
    ]
    .copy()
)


english_validation_df = (
    generic_evaluation_df[
        generic_evaluation_df["circular_id"].isin(
            validation_ids
        )
        &
        (
            generic_evaluation_df[
                "target_language"
            ] == "English"
        )
    ]
    .copy()
)


X_en_train = (
    english_train_df[
        improved_feature_columns
    ]
)

y_en_train = (
    english_train_df[
        "synthetic_missing"
    ]
    .astype(int)
)


X_en_validation = (
    english_validation_df[
        improved_feature_columns
    ]
)

y_en_validation = (
    english_validation_df[
        "synthetic_missing"
    ]
    .astype(int)
)


print(
    "English train:",
    X_en_train.shape
)

print(
    "English validation:",
    X_en_validation.shape
)

English train: (2748, 7)
English validation: (639, 7)


In [154]:
english_models = {
    "Logistic Regression": Pipeline(
        [
            (
                "scaler",
                StandardScaler()
            ),
            (
                "classifier",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=1000,
                    random_state=42
                )
            )
        ]
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "SVM (RBF)": Pipeline(
        [
            (
                "scaler",
                StandardScaler()
            ),
            (
                "classifier",
                SVC(
                    kernel="rbf",
                    class_weight="balanced",
                    probability=True,
                    random_state=42
                )
            )
        ]
    ),

    "HistGradientBoosting":
        HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.05,
            max_depth=6,
            class_weight="balanced",
            random_state=42
        )
}

In [155]:
english_improved_results = []

trained_english_models = {}


for model_name, current_model in english_models.items():

    current_model.fit(
        X_en_train,
        y_en_train
    )

    predictions = (
        current_model.predict(
            X_en_validation
        )
    )

    english_improved_results.append(
        {
            "model": model_name,

            "accuracy": accuracy_score(
                y_en_validation,
                predictions
            ),

            "precision": precision_score(
                y_en_validation,
                predictions,
                zero_division=0
            ),

            "recall": recall_score(
                y_en_validation,
                predictions,
                zero_division=0
            ),

            "f1_score": f1_score(
                y_en_validation,
                predictions,
                zero_division=0
            )
        }
    )

    trained_english_models[
        model_name
    ] = current_model


english_improved_results_df = (
    pd.DataFrame(
        english_improved_results
    )
    .sort_values(
        "f1_score",
        ascending=False
    )
)

english_improved_results_df

,model,accuracy,precision,recall,f1_score
3,HistGradientBoosting,0.704225,0.320225,0.456,0.376238
0,Logistic Regression,0.665102,0.294931,0.512,0.374269
2,SVM (RBF),0.727700,0.331034,0.384,0.355556
1,Random Forest,0.726135,0.284483,0.264,0.273859


In [156]:
#ADD threshold tuning for English HistGradientBoosting
best_english_model = (
    trained_english_models[
        "HistGradientBoosting"
    ]
)

english_validation_probabilities = (
    best_english_model
    .predict_proba(
        X_en_validation
    )[:, 1]
)

In [157]:
english_threshold_results = []

for threshold in np.arange(
    0.20,
    0.76,
    0.05
):

    predictions = (
        english_validation_probabilities
        >= threshold
    ).astype(int)

    english_threshold_results.append(
        {
            "threshold": round(
                threshold,
                2
            ),

            "accuracy": accuracy_score(
                y_en_validation,
                predictions
            ),

            "precision": precision_score(
                y_en_validation,
                predictions,
                zero_division=0
            ),

            "recall": recall_score(
                y_en_validation,
                predictions,
                zero_division=0
            ),

            "f1_score": f1_score(
                y_en_validation,
                predictions,
                zero_division=0
            )
        }
    )


english_threshold_df = (
    pd.DataFrame(
        english_threshold_results
    )
    .sort_values(
        "f1_score",
        ascending=False
    )
)

english_threshold_df

,threshold,accuracy,precision,recall,f1_score
4,0.40,0.640063,0.290837,0.584,0.388298
7,0.55,0.727700,0.341935,0.424,0.378571
3,0.35,0.572770,0.264331,0.664,0.378132
6,0.50,0.704225,0.320225,0.456,0.376238
5,0.45,0.658842,0.287671,0.504,0.366279
2,0.30,0.500782,0.242021,0.728,0.363273
1,0.25,0.442879,0.230769,0.792,0.357401
0,0.20,0.402191,0.222462,0.824,0.350340
8,0.60,0.740219,0.330579,0.320,0.325203
9,0.65,0.762128,0.363636,0.288,0.321429


In [158]:
#Find completely unused circulars
used_circular_ids = set(
    generic_evaluation_ids
) | set(
    evaluation_ids
)

fresh_holdout_ids = [
    circular_id
    for circular_id in eligible_circulars.index
    if circular_id not in used_circular_ids
]

print(
    "Fresh holdout circulars:",
    len(fresh_holdout_ids)
)

print(
    fresh_holdout_ids
)

Fresh holdout circulars: 24
[150, 786, 1059, 1202, 1334, 1496, 1516, 1530, 1536, 1547, 1589, 1631, 1690, 1765, 1802, 1844, 2005, 2090, 2099, 2100, 2126, 2161, 2168, 2190]


In [159]:
#Generate English-only fresh holdout cases
english_holdout_cases = []

for index, circular_id in enumerate(
    fresh_holdout_ids,
    start=1
):

    print(
        f"Processing {index}/"
        f"{len(fresh_holdout_ids)}"
        f" - Circular {circular_id}"
    )

    try:

        prepared = (
            prepare_multilingual_circular(
                circular_id
            )
        )

        english_count = len(
            prepared["English"]["chunks"]
        )

        scenarios = (
            create_missing_scenarios(
                english_count
            )
        )

        for (
            scenario_name,
            positions
        ) in scenarios.items():

            case_df = build_generic_missing_case(
                prepared=prepared,
                target_language="English",
                remove_positions=positions,
                scenario_name=scenario_name,
                circular_id=circular_id
            )

            english_holdout_cases.append(
                case_df
            )

    except Exception as error:

        print(
            "Failed:",
            circular_id,
            error
        )

Processing 1/24 - Circular 150
Processing 2/24 - Circular 786
Processing 3/24 - Circular 1059
Processing 4/24 - Circular 1202
Processing 5/24 - Circular 1334
Processing 6/24 - Circular 1496
Processing 7/24 - Circular 1516
Processing 8/24 - Circular 1530
Processing 9/24 - Circular 1536
Processing 10/24 - Circular 1547
Processing 11/24 - Circular 1589
Processing 12/24 - Circular 1631
Processing 13/24 - Circular 1690
Processing 14/24 - Circular 1765
Processing 15/24 - Circular 1802
Processing 16/24 - Circular 1844
Processing 17/24 - Circular 2005
Processing 18/24 - Circular 2090
Processing 19/24 - Circular 2099
Processing 20/24 - Circular 2100
Processing 21/24 - Circular 2126
Processing 22/24 - Circular 2161
Processing 23/24 - Circular 2168
Processing 24/24 - Circular 2190


In [160]:
#Combine
english_holdout_df = pd.concat(
    english_holdout_cases,
    ignore_index=True
)

print(
    "Holdout circulars:",
    english_holdout_df[
        "circular_id"
    ].nunique()
)

print(
    "Holdout rows:",
    len(english_holdout_df)
)

print(
    "\nClass distribution:"
)

print(
    english_holdout_df[
        "synthetic_missing"
    ].value_counts()
)

Holdout circulars: 24
Holdout rows: 1059

Class distribution:
synthetic_missing
False    839
True     220
Name: count, dtype: int64


In [161]:
X_english_holdout = (
    english_holdout_df[
        improved_feature_columns
    ]
)

y_english_holdout = (
    english_holdout_df[
        "synthetic_missing"
    ]
    .astype(int)
)


english_holdout_probabilities = (
    best_english_model
    .predict_proba(
        X_english_holdout
    )[:, 1]
)


english_holdout_predictions = (
    english_holdout_probabilities
    >= 0.40
).astype(int)

In [162]:
print(
    "Fresh English Accuracy:",
    round(
        accuracy_score(
            y_english_holdout,
            english_holdout_predictions
        ),
        4
    )
)

print(
    "Fresh English Precision:",
    round(
        precision_score(
            y_english_holdout,
            english_holdout_predictions,
            zero_division=0
        ),
        4
    )
)

print(
    "Fresh English Recall:",
    round(
        recall_score(
            y_english_holdout,
            english_holdout_predictions,
            zero_division=0
        ),
        4
    )
)

print(
    "Fresh English F1-score:",
    round(
        f1_score(
            y_english_holdout,
            english_holdout_predictions,
            zero_division=0
        ),
        4
    )
)

print(
    "\nConfusion Matrix:"
)

print(
    confusion_matrix(
        y_english_holdout,
        english_holdout_predictions
    )
)

Fresh English Accuracy: 0.678
Fresh English Precision: 0.3742
Fresh English Recall: 0.8182
Fresh English F1-score: 0.5136

Confusion Matrix:
[[538 301]
 [ 40 180]]


## Final Deployment Models

In [163]:
deployment_ids = list(
    set(train_ids)
    | set(validation_ids)
)

deployment_df = (
    generic_evaluation_df[
        generic_evaluation_df[
            "circular_id"
        ].isin(
            deployment_ids
        )
    ]
    .copy()
)

print(
    "Deployment circulars:",
    deployment_df[
        "circular_id"
    ].nunique()
)

Deployment circulars: 85


In [164]:
#English deployment model — 7 improved features
english_deployment_df = (
    deployment_df[
        deployment_df[
            "target_language"
        ] == "English"
    ]
)

X_english_deployment = (
    english_deployment_df[
        improved_feature_columns
    ]
)

y_english_deployment = (
    english_deployment_df[
        "synthetic_missing"
    ]
    .astype(int)
)


english_deployment_model = (
    HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_depth=6,
        class_weight="balanced",
        random_state=42
    )
)

english_deployment_model.fit(
    X_english_deployment,
    y_english_deployment
)

,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.05
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",200
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",6
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""category"" are considered to be categorical features. The input must be an object exposing a ``__dataframe__`` method such as pandas or polars DataFrames to use this feature.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing values. All categorical values areconverted to floating point numbers. This means that categorical valuesof 1.0 and 1 are treated as the same category.Read more in the :ref:`User Guide `... versionadded:: 0.24.. versionchanged:: 1.2 Added support for feature names... versionchanged:: 1.4 Added `""from_dtype""` option... versionchanged:: 1.6 The default value changed from `None` to `""from_dtype""`.",'from_dtyp

In [165]:
#Sinhala + Tamil deployment features
standard_feature_columns = [
    "target_similarity",
    "reference_similarity",
    "cross_language_gap",
    "compression_count",
    "local_drop"
]

In [166]:
#Sinhala:
sinhala_deployment_df = (
    deployment_df[
        deployment_df[
            "target_language"
        ] == "Sinhala"
    ]
)

sinhala_deployment_model = Pipeline(
    [
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            SVC(
                kernel="rbf",
                class_weight="balanced",
                probability=True,
                random_state=42
            )
        )
    ]
)

sinhala_deployment_model.fit(
    sinhala_deployment_df[
        standard_feature_columns
    ],
    sinhala_deployment_df[
        "synthetic_missing"
    ].astype(int)
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'


In [167]:
#Tamil:
tamil_deployment_df = (
    deployment_df[
        deployment_df[
            "target_language"
        ] == "Tamil"
    ]
)

tamil_deployment_model = (
    RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
)

tamil_deployment_model.fit(
    tamil_deployment_df[
        standard_feature_columns
    ],
    tamil_deployment_df[
        "synthetic_missing"
    ].astype(int)
)

print(
    "All three deployment models trained."
)

All three deployment models trained.


In [168]:
#Save final 3 language models
import os
import json
import joblib


# Create models folder if it does not exist
os.makedirs(
    "../models",
    exist_ok=True
)


# Save English detector
joblib.dump(
    english_deployment_model,
    "../models/tricheck_english_detector.joblib"
)


# Save Sinhala detector
joblib.dump(
    sinhala_deployment_model,
    "../models/tricheck_sinhala_detector.joblib"
)


# Save Tamil detector
joblib.dump(
    tamil_deployment_model,
    "../models/tricheck_tamil_detector.joblib"
)


print(
    "All three detector models saved."
)

All three detector models saved.


In [169]:
#Save final configuration
deployment_config = {

    "embedding_model":
        "intfloat/multilingual-e5-small",

    "languages": {

        "English": {
            "model":
                "HistGradientBoosting",

            "threshold":
                0.40,

            "features":
                improved_feature_columns
        },

        "Sinhala": {
            "model":
                "SVM (RBF)",

            "threshold":
                0.40,

            "features":
                standard_feature_columns
        },

        "Tamil": {
            "model":
                "Random Forest",

            "threshold":
                0.50,

            "features":
                standard_feature_columns
        }
    }
}


with open(
    "../models/tricheck_multilingual_config.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        deployment_config,
        file,
        indent=4
    )


print(
    "TriCheck-LK multilingual configuration saved."
)

TriCheck-LK multilingual configuration saved.


In [170]:
#quick verification
print(
    json.dumps(
        deployment_config,
        indent=4
    )
)

{
    "embedding_model": "intfloat/multilingual-e5-small",
    "languages": {
        "English": {
            "model": "HistGradientBoosting",
            "threshold": 0.4,
            "features": [
                "target_similarity",
                "reference_similarity",
                "cross_language_gap",
                "compression_count",
                "local_drop",
                "position_difference",
                "compression_local_max"
            ]
        },
        "Sinhala": {
            "model": "SVM (RBF)",
            "threshold": 0.4,
            "features": [
                "target_similarity",
                "reference_similarity",
                "cross_language_gap",
                "compression_count",
                "local_drop"
            ]
        },
        "Tamil": {
            "model": "Random Forest",
            "threshold": 0.5,
            "features": [
                "target_similarity",
                "reference_similarity",
       